# **Validación y explotación del modelo Corepulse**

Este notebook documenta el proceso posterior a la preparación del modelo dimensional del proyecto **Corepulse Sales Analytics**.

La fase anterior dejó preparados los scripts de transformación, las tablas finales del modelo y el script SQL de carga en Snowflake. En esta fase se valida que el modelo se haya cargado correctamente en base de datos y se preparan las bases para su explotación en herramientas de Business Intelligence.


## **1. Objetivo de esta fase**.

El objetivo principal de esta fase es comprobar que el modelo estrella cargado en Snowflake es consistente y está listo para ser conectado a herramientas como Power BI o Looker Studio.

Las validaciones se centran en:

- Comprobar que las tablas se han creado y cargado correctamente.
- Validar que las dimensiones no tienen claves duplicadas.
- Comprobar que la tabla de hechos no contiene claves huérfanas.
- Validar que la granularidad de la fact se mantiene correctamente.
- Dejar documentado el estado del modelo antes de construir dashboards.


---

## **2. Modelo cargado en Snowflake**.

El modelo sigue una estructura de **modelo estrella**, con una tabla de hechos central y varias dimensiones descriptivas.

### Tabla de hechos

- `fact_ventas`

### Dimensiones

- `dim_producto`
- `dim_categoria`
- `dim_proveedor`
- `dim_calendario`

La granularidad esperada de la fact es:

```text
1 fila = 1 producto + 1 semana de negocio
```

Esto significa que la combinación `product_id + year_week_key` debe ser única dentro de `fact_ventas`.


----

## **3. Validación de los datos cargados.**

### **3.1. Validaciones: conteo de filas por tabla.**

La primera comprobación sirve para confirmar que todas las tablas se han cargado y que el volumen de registros es coherente con lo esperado.

Esta validación no comprueba todavía relaciones ni duplicados; simplemente responde a la pregunta:

> ¿Se han creado y cargado las tablas principales del modelo?


```yaml
-- Conteo de filas por tabla
SELECT 'dim_categoria' AS tabla, COUNT(*) AS filas FROM dim_categoria
UNION ALL
SELECT 'dim_proveedor', COUNT(*) FROM dim_proveedor
UNION ALL
SELECT 'dim_producto', COUNT(*) FROM dim_producto
UNION ALL
SELECT 'dim_calendario', COUNT(*) FROM dim_calendario
UNION ALL
SELECT 'fact_ventas', COUNT(*) FROM fact_ventas;

### Resultado obtenido

| Tabla | Filas |
|---|---:|
| dim_categoria | 19 |
| dim_proveedor | 20 |
| dim_producto | 60 |
| dim_calendario | 159 |
| fact_ventas | 9540 |

### Interpretación

El modelo se ha cargado correctamente en Snowflake. La fact contiene 9540 registros, coherentes con una estructura semanal por producto.

La lectura de la fact es coherente con el grano esperado:

```text
60 productos × 159 semanas = 9540 filas
```

Por tanto, a nivel de volumen, la carga parece correcta.


### **3.2. Validación 2: duplicados en dimensiones.**

Las dimensiones deben tener una fila única por cada clave primaria lógica.

Aunque en Snowflake se pueden declarar claves primarias y foráneas, en tablas estándar estas restricciones funcionan principalmente como metadatos. Por eso es importante comprobar manualmente que las claves realmente se comportan como únicas.

En el caso de `dim_calendario`, cada `year_week_key` debe aparecer una sola vez.


```yaml

-- Duplicados en dim_calendario
SELECT 
    year_week_key,
    COUNT(*) AS n_filas
FROM dim_calendario
GROUP BY year_week_key
HAVING COUNT(*) > 1;


### Lógica de la query

- `GROUP BY year_week_key` agrupa todas las filas que pertenecen a la misma semana.
- `COUNT(*)` cuenta cuántas filas hay para cada semana.
- `HAVING COUNT(*) > 1` muestra únicamente las semanas que aparecen más de una vez.

La diferencia clave es:

```text
WHERE  → filtra filas antes de agrupar
HAVING → filtra grupos después de agrupar
```

En este caso se usa `HAVING` porque queremos filtrar grupos agregados, no filas individuales.

### Resultado obtenido

La consulta devuelve **0 filas**.

### Interpretación

No hay semanas duplicadas en `dim_calendario`. La clave `year_week_key` funciona correctamente como identificador único de semana.


### **3.3. Validación 3: claves huérfanas entre fact y calendario.**

Esta validación comprueba si existen registros en `fact_ventas` cuya semana no exista en `dim_calendario`.

Una clave huérfana aparecería si la fact contiene un `year_week_key` que no tiene correspondencia en la dimensión calendario.


```yaml

-- Claves huérfanas entre fact_ventas y dim_calendario
SELECT COUNT(*) AS fact_sin_calendario
FROM fact_ventas f
LEFT JOIN dim_calendario c
    ON f.year_week_key = c.year_week_key
WHERE c.year_week_key IS NULL;

### Lógica de la query

Se utiliza un `LEFT JOIN` desde `fact_ventas` hacia `dim_calendario`.

Esto conserva todas las filas de la fact. Si una fila de la fact no encuentra coincidencia en calendario, las columnas de `dim_calendario` quedan como `NULL`.

Por eso se filtra con:

```sql
WHERE c.year_week_key IS NULL
```

Esa condición identifica las filas de la fact que no han encontrado correspondencia en la dimensión.

### Resultado obtenido

`fact_sin_calendario = 0`

### Interpretación

Todas las semanas presentes en `fact_ventas` existen también en `dim_calendario`. No hay claves huérfanas en la relación temporal.


### **3.4. Validación 4: granularidad de la fact.**

La granularidad esperada de `fact_ventas` es:

```text
1 fila = 1 producto + 1 semana de negocio
```

Por tanto, no debe existir más de una fila para la misma combinación de `product_id` y `year_week_key`.


```yaml

-- Validación de granularidad de la fact
SELECT 
    product_id,
    year_week_key,
    COUNT(*) AS n_filas
FROM fact_ventas
GROUP BY product_id, year_week_key
HAVING COUNT(*) > 1;

### Lógica de la query

- `GROUP BY product_id, year_week_key` agrupa por cada combinación producto-semana.
- `COUNT(*)` cuenta cuántas filas existen para cada combinación.
- `HAVING COUNT(*) > 1` filtra únicamente las combinaciones repetidas.

Si esta query devolviera resultados, significaría que la fact tiene duplicados a nivel de grano.

### Resultado obtenido

La consulta devuelve **0 filas**.

### Interpretación

La fact respeta correctamente la granularidad definida. No existen duplicados por producto y semana.


### **3.5. Resumen de validaciones.**

| Validación | Resultado | Interpretación |
|---|---:|---|
| Conteo de tablas | Correcto | Las tablas se han cargado correctamente. |
| Duplicados en calendario | 0 filas | `year_week_key` es único en `dim_calendario`. |
| Fact sin calendario | 0 | Todas las semanas de la fact existen en calendario. |
| Duplicados producto-semana | 0 filas | La fact respeta su granularidad. |

Conclusión: el modelo cargado en Snowflake está correctamente estructurado y puede utilizarse como base para la fase de visualización.


----

## 8. Próximos pasos

Una vez validado el modelo en Snowflake, los siguientes pasos del proyecto son:

1. Conectar Snowflake con Power BI.
2. Revisar relaciones del modelo en Power BI.
3. Crear medidas DAX principales.
4. Diseñar las páginas del dashboard.
5. Replicar o adaptar el análisis en Looker Studio.
6. Documentar decisiones de diseño y visualización.

Este notebook se actualizará progresivamente conforme avance la fase de explotación analítica.


----

## **4. Diseño de la capa analítica en Snowflake.**

Una vez cargado y validado el modelo estrella en Snowflake, el siguiente paso consiste en decidir qué cálculos deben construirse directamente en la base de datos y cuáles deben dejarse para la herramienta de visualización.

Esta decisión es importante porque no todos los cálculos tienen la misma naturaleza:

- Algunos cálculos son estables y reutilizables.
- Otros dependen del contexto de filtros del usuario.
- Algunos deben servir tanto para Power BI como para Looker Studio.
- Otros solo tienen sentido dentro de una visualización concreta.

Por este motivo, se plantea una separación entre:

```text
Snowflake       → capa analítica reutilizable
Power BI        → métricas dinámicas dependientes del contexto visual
Looker Studio   → visualización y campos calculados simples

### **4.1. Cálculos que se preparan en Snowflake**. 



En Snowflake se prepararán aquellas lógicas que interesa reutilizar en distintas herramientas de BI o que resultan más complejas de mantener directamente en la capa visual.

Ejemplos:
- vistas enriquecidas con joins entre fact y dimensiones;
- rankings anuales de productos;
- rankings anuales de proveedores;
- clasificaciones de productos;
- clasificaciones de proveedores;
- comparativas anuales;
- cálculos base de evolución YoY;
- rankings previstos para escenarios de forecast.



### **4.2. Cálculos que se dejan en Power BI**.


Power BI se utilizará para cálculos que deben responder dinámicamente a los filtros del usuario.

Ejemplos:
- ranking dinámico según categoría seleccionada;
- ranking dinámico según proveedor;
- top N dinámico;
- contribución de productos dentro del contexto filtrado;
- medidas afectadas por slicers;
- variaciones según selección temporal;
- KPIs interactivos.

Esto se debe a que DAX recalcula las medidas en función del contexto visual. Por tanto, un ranking creado en Power BI con RANKX puede cambiar cuando el usuario filtra por categoría, proveedor, campaña o periodo.

### **4.3. Diferencia entre ranking fijo y ranking dinámico.** 

Un ranking calculado en Snowflake es un ranking fijo o precalculado.

Por ejemplo, si se calcula el ranking anual de productos para 2025, Snowflake ordena los productos según las ventas de ese año y guarda esa posición en una vista.

Este ranking no cambia automáticamente si después, en Power BI o Looker Studio, se filtra por una categoría concreta. El ranking seguirá reflejando la posición calculada en el contexto original definido en SQL.

Por ejemplo, 

Ranking global 2025:

Producto A → ranking 1

Producto B → ranking 2

Producto C → ranking 3

Si después se filtra una categoría en Power BI, el producto C podría seguir mostrando ranking 3 aunque dentro de esa categoría sea el segundo producto visible.


### **4.4. Uso de PARTITION BY en rankings anuales.** 

Para calcular rankings separados por año en Snowflake se utiliza PARTITION BY.

La lógica es:

```yaml

RANK() OVER (
    PARTITION BY year_label
    ORDER BY sales_value DESC
) AS ranking_producto_anual

Esto significa que el ranking se reinicia para cada año y, por tanto, snowflake calcula de forma independiente y sin necesidad de crear tres consultas distintas:

Ranking de productos 2024

Ranking de productos 2025

Ranking de productos 2026

### **4.5. Tratamiento de escenarios de forecast**.

En el modelo existen ventas reales para 2024 y 2025, pero para 2026 existen previsiones en tres escenarios:

- forecast base,
- forecast optimista, 
- forecast pesimista.

Por tanto, si se desea analizar el ranking previsto de productos en 2026, este debe calcularse por escenario. 

La opción más limpia es transformar las métricas en formato largo, creando una columna escenario. Para conseguirlo, la query debe calcular el ranking con:

```yaml

PARTITION BY year_label, escenario

Esto permite obtener un ranking independiente para cada combinación de año y escenario.

### **4.6. Por qué no se crean tres vistas separadas para 2026.**

Aunque sería posible crear una vista para cada escenario, no se considera la opción más mantenible. 

Crear una única vista en formato largo es más escalable porque:

- evita duplicar lógica, 
- permite filtrar por escenario en la herramienta BI, 
- facilita la comparación entre escenarios, 
- permite reutilizar la misma estructura para productos y proveedores y
- simplifica el mantenimiento del modelo. 

Por tanto, se prefiere una vista única con una columna escenario. 

### **4.7. Decisión final.**

La decisión adoptada es crear en Snowflake una capa analítica con rankings fijos y comparativas base, y dejar los rankings dinámicos para Power BI.

En Snowflake se crearán vistas como:

vw_ventas_base
vw_ranking_productos_anual_escenario
vw_ranking_proveedores_anual_escenario
vw_comparativa_ranking_productos_anual
vw_comparativa_ranking_proveedores_anual

En Power BI se crearán medidas DAX para los cálculos que deban cambiar según los filtros del usuario.

Esta separación permite construir un modelo más profesional, reutilizable y claro. 

### **4.8. Explicación de las vistas.**

#### ➡️ **Vista 1 - vw_ventas_base.**

Es la vista base de consumo. Esto es, una vista que una la tabla de hechos con las dimensiones para no tener que repetir joins en cada análisis. 

```yaml

CREATE OR REPLACE VIEW vw_ventas_base AS
SELECT
    f.product_id,
    f.category_id,
    f.provider_id,
    f.year_week_key,

    p.nombre,
    p.marca,
    p.cluster_final,
    p.perfil_comportamiento,
    p.unit_sale_price_reference,

    c.categoria,
    c.familia_categoria,
    c.formato_categoria,

    pr.proveedor,
    pr.pais,
    pr.ccaa,
    pr.tipo_proveedor,
    pr.lead_time_dias_sim,
    pr.pedido_minimo_sim,

    cal.year_label,
    cal.week_number_business,
    cal.week_start_date,
    cal.week_end_date,
    cal.week_label,
    cal.month_start_name,
    cal.quarter_start_label,
    cal.is_partial_week,
    cal.is_campaign_week,
    cal.campaign_name_primary,
    cal.campaign_group_primary,

    f.sales_units,
    f.forecast_base_units,
    f.forecast_optimista_units,
    f.forecast_pesimista_units,
    f.sales_value_estimated,
    f.forecast_base_value_estimated,
    f.forecast_optimista_value_estimated,
    f.forecast_pesimista_value_estimated,
    f.sales_units_adjusted,
    f.forecast_base_units_adjusted,
    f.forecast_optimista_units_adjusted,
    f.forecast_pesimista_units_adjusted,
    f.sales_value_estimated_adjusted,
    f.forecast_base_value_estimated_adjusted,
    f.forecast_optimista_value_estimated_adjusted,
    f.forecast_pesimista_value_estimated_adjusted

FROM fact_ventas f
LEFT JOIN dim_producto p
    ON f.product_id = p.product_id
LEFT JOIN dim_categoria c
    ON f.category_id = c.category_id
LEFT JOIN dim_proveedor pr
    ON f.provider_id = pr.provider_id
LEFT JOIN dim_calendario cal
    ON f.year_week_key = cal.year_week_key;


🟪 **1. Qué problema resuelve.**

La vista `vw_ventas_base` se crea para centralizar en una única consulta la unión entre la tabla de hechos `fact_ventas` y sus dimensiones principales.

Aunque el modelo está correctamente estructurado en formato estrella, para determinados análisis y herramientas de visualización puede ser útil disponer de una vista enriquecida que ya incluya los atributos descriptivos de producto, categoría, proveedor y calendario.

Esta vista evita tener que repetir los mismos `JOIN` en cada análisis posterior y funciona como una capa de consumo analítico sobre el modelo dimensional.


🟪 **2. Grano del resultado.**

La vista conserva la granularidad de la tabla de hechos:

**1 fila = 1 producto + 1 semana de negocio**


🟪 **3. Columnas principales generadas.**

La vista incorpora varios bloques de informacion:

- Claves del modelo.
    - product_id
    - category_id
    - provider_id
    - year_week_key

- Atributos de producto.
    - nombre
    - marca
    - clúster
    - perfil de comportamiento
    - precio unitario de referencia

- Atributos de categoría.
    - categoría
    - familia de categoría
    - formato de la categoría

- Atributos de proveedor.
    - proveedor
    - país
    - comunidad autónoma
    - tipo de proveedor
    - lead time simulado
    - pedido mínimo simulado

- Atributos temporales. 
    - año
    - número de sema
    - fecha de inicio y fin de semana
    - campaña principal
    - flag de semana parcial
    - flag de semana de campaña

- Métricas. 
    - unidades vendidas reales
    - forecast base
    - forecast optimista
    - forecast pesimista
    - valores estimados
    - métricas ajustadas para semanas parciales

🟪 **4. Lógica utilizada.**

La query parte de fact_ventas, que es la tabla central del modelo, y utiliza LEFT JOIN para incorporar los atributos de cada dimensión.

Se usa LEFT JOIN porque se quiere conservar siempre la totalidad de registros de la fact. Si alguna dimensión no encontrara correspondencia, la fila de la fact seguiría apareciendo, permitiendo detectar posibles problemas de claves huérfanas.

La estructura principal es:

```yaml
fact_ventas
    LEFT JOIN dim_producto
    LEFT JOIN dim_categoria
    LEFT JOIN dim_proveedor
    LEFT JOIN dim_calendario




🟪 **5. Decisión técnica.**

Aunque el modelo estrella se mantiene como estructura principal, se crea esta vista como una capa analítica adicional. Esta decisión permite separar dos niveles:

- Tablas base del modelo estrella → estructura relacional limpia
- Vista de consumo → tabla enriquecida para análisis y visualización

La vista vw_ventas_base es equivalente, conceptualmente, a una capa de exploración o consumo sobre el modelo. En herramientas como Looker podría asimilarse a una vista/explore base; en Snowflake se implementa directamente como una VIEW.

🟪 **6. Limitaciones.**

La vista depende de que las dimensiones tengan claves únicas. Si alguna dimensión tuviera duplicados en sus claves, los JOIN podrían multiplicar filas y alterar las métricas. 

Por este motivo, antes de utilizar esta vista se han llevado a cabo las siguientes validaciones:

- duplicados en dimensiones, 
- claves huérfanas y 
- granularidad de la fact. 

Además, aunque esta vista es cómoda para herramientas como Looker Studio, en Power BI puede ser preferible mantener también el modelo estrella con relaciones entre tablas para aprovechar mejor el motor semántico y las medidas DAX.

🟪 **7. Uso posterior.**

La vista vw_ventas_base servirá como base para construir otras vistas analíticas en Snowflake, como:

- rankings de productos, 
- rankings de proveedores, 
- clasificaciones, 
- análisis YoY, 
- comparativas entre escenarios, 
- vistas de consumo para Power BI o Data Studio. 

También puede utilizarse directamente como fuente en herramientas de BI cuando se quiera trabajar con una tabla ya enriquecida y preparada para visualización.

#### ➡️ **Vista 1 - vw_ventas_base.**

🟪 **1. Qué problema resuelve.**

🟪 **2. Grano del resultado.**

🟪 **3. Columnas principales generadas.**

🟪 **4. Lógica utilizada.**

🟪 **5. Decisión técnica.**

🟪 **6. Limitaciones.**

🟪 **7. Uso posterior.**

#### ➡️ **Vista 2 - vw_ranking_productos_anual_escenario.**

Con esta vista, se resuelven varias cosas a la vez:

- ranking 2024 con ventas reales, 
- ranking 2025 con ventas reales, 
- ranking 2026 con forecast base, 
- ranking 2026 con forecast optimista, 
- ranking 2026 con forecast pesimista, 
- contribución individual, 
- contribución acumulada. 


```yaml

CREATE OR REPLACE VIEW vw_ranking_productos_anual_escenario AS

WITH metricas_producto_anual AS (

    -- Ventas reales 2024 y 2025
    SELECT
        year_label,
        'Real' AS escenario,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor,
        SUM(sales_units) AS units_value,
        SUM(sales_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label IN (2024, 2025)
    GROUP BY
        year_label,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor

    UNION ALL

    -- Forecast 2026: escenario base
    SELECT
        year_label,
        'Forecast base' AS escenario,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor,
        SUM(forecast_base_units) AS units_value,
        SUM(forecast_base_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor

    UNION ALL

    -- Forecast 2026: escenario optimista
    SELECT
        year_label,
        'Forecast optimista' AS escenario,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor,
        SUM(forecast_optimista_units) AS units_value,
        SUM(forecast_optimista_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor

    UNION ALL

    -- Forecast 2026: escenario pesimista
    SELECT
        year_label,
        'Forecast pesimista' AS escenario,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor,
        SUM(forecast_pesimista_units) AS units_value,
        SUM(forecast_pesimista_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        product_id,
        nombre,
        marca,
        categoria,
        proveedor
),

ranking AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY year_label, escenario
            ORDER BY sales_value DESC
        ) AS ranking_producto,

        sales_value
        / NULLIF(
            SUM(sales_value) OVER (
                PARTITION BY year_label, escenario
            ),
            0
        ) AS pct_contribucion,

        SUM(sales_value) OVER (
            PARTITION BY year_label, escenario
            ORDER BY sales_value DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
        / NULLIF(
            SUM(sales_value) OVER (
                PARTITION BY year_label, escenario
            ),
            0
        ) AS pct_contribucion_acumulada

    FROM metricas_producto_anual
)

SELECT *
FROM ranking;

🟪 **1. Qué problema resuelve.**

La vista `vw_ranking_productos_anual_escenario` se crea para calcular el ranking anual de productos teniendo en cuenta tanto los años con ventas reales como el año forecast.

El proyecto contiene:

- ventas reales para 2024;
- ventas reales para 2025;
- forecast para 2026 en tres escenarios: base, optimista y pesimista.

Por tanto, no basta con calcular un ranking único para 2026. Es necesario calcular un ranking independiente para cada escenario de forecast, ya que la posición de los productos puede variar según el escenario analizado.

Esta vista permite responder preguntas como:

- qué productos lideran las ventas reales en 2024;
- qué productos lideran las ventas reales en 2025;
- qué productos lideran el forecast base de 2026;
- si el ranking cambia en el escenario optimista;
- si el ranking cambia en el escenario pesimista;
- qué productos concentran mayor contribución sobre el total anual.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:

1 fila = 1 producto + 1 año + 1 escenario


🟪 **3. Columnas principales generadas.**

La vista incluye:

- Identificación temporal y de escenario:
    - year_label
    - escenario

- Identificación del producto:
    - product_id
    - nombre
    - marca
    - categoria
    - proveedor

- Métricas agregadas:
    - units_value
    - sales_value

- Métricas analíticas: 
    - ranking_producto
    - pct_contribucion
    - pct_contribucion_acumulada

🟪 **4. Lógica utilizada.**

La query parte de la vista vw_ventas_base, que ya contiene la fact enriquecida con atributos de producto, categoría, proveedor y calendario.

El primer bloque, metricas_producto_anual, transforma las métricas reales y de forecast en una estructura común.

Para 2024 y 2025 se utilizan las ventas reales:

- SUM(sales_units)
- SUM(sales_value_estimated)

Para 2026 se utilizan las métricas de forecast:

- SUM(forecast_base_units)
- SUM(forecast_base_value_estimated)

- SUM(forecast_optimista_units)
- SUM(forecast_optimista_value_estimated)

- SUM(forecast_pesimista_units)
- SUM(forecast_pesimista_value_estimated)

Para unificar todos los casos, se crea una columna llamada escenario.

Esto permite que ventas reales y forecast tengan una misma estructura analítica.


🟪 **5. Uso de UNION ALL.**

Se usa UNION ALL para apilcar los distintos bloques de datos:

Ventas reales 2024-2025
+
Forecast base 2026
+
Forecast optimista 2026
+
Forecast pesimista 2026

La ventaja de este enfoque es que evita crear tres vistas separadas para cada escenario de 2026. Esto hace que la solución sea más mantenible y más fácil de explotar desde Power BI o Looker Studio.

🟪 **6. Uso de PARTITION BY.**

El ranking se calcula con:

```yaml

RANK() OVER (
    PARTITION BY year_label, escenario
    ORDER BY sales_value DESC
) AS ranking_producto

La cláusula PARTITION BY year_label, escenario indica que el ranking debe reiniciarse para cada combinación de año y escenario.

Es decir, Snowflake calcula rankings independientes para:

```yaml
2024 - Real
2025 - Real
2026 - Forecast base
2026 - Forecast optimista
2026 - Forecast pesimista


Sin PARTITION BY, Snowflake calcularía un ranking global mezclando años y escenarios, lo cual no tendría sentido analítico.

🟪 **7. Contribución individual.**

La contribución individual indica qué porcentaje representa cada producto sobre le total de ventas o forecast de su año y escenario. 

Por ejemplo, si un producto representa el 8% del total del escenario base 2026, su pct_contribucion será 0,08. 

Se usa NULLIF(...,0) para evitar divisiones entre cero. 

Se calcula como sigue:

```yaml
sales_value
/ NULLIF(
    SUM(sales_value) OVER (
        PARTITION BY year_label, escenario
    ),
    0
) AS pct_contribucion


🟪 **8. Contribución acumulada.**

La contribución acumulada se calcula ordenando los productos de mayor a menor valor:

```yaml
SUM(sales_value) OVER (
    PARTITION BY year_label, escenario
    ORDER BY sales_value DESC
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW #suma desde la primera fila del ranking hasta la fila actual. 
)
/
NULLIF(
    SUM(sales_value) OVER (
        PARTITION BY year_label, escenario
    ),
    0
) AS pct_contribucion_acumulada

Esta métrica permite analizar cuánto peso acumulan los productos según su posición en el ranking.

Es útil para identificar:
- productos tops,
- productos core,
- productos secundarios, 
- long tail, 
- concentración de ventas. 

🟪 **9. Decisión técnica.**

La decisión adoptada es crear el ranking de productos en Snowflake como una vista fija y reutilizable.

Esta vista no pretende sustituir a los rankings dinámicos que puedan crearse posteriormente en Power BI. Su objetivo es generar una capa analítica estable para comparar años y escenarios de forma consistente.

La vista es especialmente útil para Looker Studio, donde las opciones de cálculo dinámico son más limitadas que en Power BI.

🟪 **10. Limitaciones.**

El ranking generado en esta vista es un ranking fijo o precalculado. Esto significa que el ranking no se recalcula automáticamente si en Power BI o Looker Studio se aplican filtros adicionales no contemplados en la partición.

Por ejemplo, si el ranking se calcula a nivel global por año y escenario, y después el usuario filtra una categoría concreta, el ranking seguirá mostrando la posición global del producto, no su posición dentro de esa categoría.

Para rankings completamente dinámicos se utilizarán medidas DAX en Power BI.


🟪 **11. Uso posterior.**

Esta vista se utilizará para:

- tablas de ranking anual de productos, 
- análisis top de productos por año, 
- comparación entre ventas reales y forecast, 
- análisis de escenarios de 2026, 
- cáclulo de clasificaciones de producto, 
- creación de vistas comparativas entre años. 

Además, servirá como base para construir vistas posteriores, como vw_comparativa_ranking_productos_anual.

#### ➡️ **Vista 3 - vw_ranking_proveedores_anual_escenario.**

Sirve para saber qué proveedores concentran más ventas reales o previstas según el año y el escenario. 


```yaml
CREATE OR REPLACE VIEW vw_ranking_proveedores_anual_escenario AS

WITH metricas_proveedor_anual AS (

    -- Ventas reales 2024 y 2025
    SELECT
        year_label,
        'Real' AS escenario,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim,
        COUNT(DISTINCT product_id) AS n_productos,
        SUM(sales_units) AS units_value,
        SUM(sales_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label IN (2024, 2025)
    GROUP BY
        year_label,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim

    UNION ALL

    -- Forecast 2026: escenario base
    SELECT
        year_label,
        'Forecast base' AS escenario,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim,
        COUNT(DISTINCT product_id) AS n_productos,
        SUM(forecast_base_units) AS units_value,
        SUM(forecast_base_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim

    UNION ALL

    -- Forecast 2026: escenario optimista
    SELECT
        year_label,
        'Forecast optimista' AS escenario,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim,
        COUNT(DISTINCT product_id) AS n_productos,
        SUM(forecast_optimista_units) AS units_value,
        SUM(forecast_optimista_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim

    UNION ALL

    -- Forecast 2026: escenario pesimista
    SELECT
        year_label,
        'Forecast pesimista' AS escenario,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim,
        COUNT(DISTINCT product_id) AS n_productos,
        SUM(forecast_pesimista_units) AS units_value,
        SUM(forecast_pesimista_value_estimated) AS sales_value
    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        provider_id,
        proveedor,
        pais,
        ccaa,
        tipo_proveedor,
        lead_time_dias_sim,
        pedido_minimo_sim
),

ranking AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY year_label, escenario
            ORDER BY sales_value DESC
        ) AS ranking_proveedor,

        sales_value
        / NULLIF(
            SUM(sales_value) OVER (
                PARTITION BY year_label, escenario
            ),
            0
        ) AS pct_contribucion,

        SUM(sales_value) OVER (
            PARTITION BY year_label, escenario
            ORDER BY sales_value DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
        / NULLIF(
            SUM(sales_value) OVER (
                PARTITION BY year_label, escenario
            ),
            0
        ) AS pct_contribucion_acumulada

    FROM metricas_proveedor_anual
)

SELECT *
FROM ranking;



🟪 **1. Qué problema resuelve.**

La vista `vw_ranking_proveedores_anual_escenario` calcula el ranking anual de proveedores combinando años con ventas reales y el año de forecast.

El objetivo es identificar qué proveedores concentran mayor volumen de ventas o previsión en cada periodo analizado.

Esta vista permite responder preguntas como:

- qué proveedores generaron más ventas en 2024;
- qué proveedores lideran en 2025;
- qué proveedores tienen mayor peso previsto en 2026;
- si el ranking de proveedores cambia según el escenario base, optimista o pesimista;
- qué proveedores concentran mayor porcentaje del valor total.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:

1 fila = 1 proveedor + 1 año + 1 escenario

🟪 **3. Columnas principales generadas.**

La vista incluye:

- Identificación temporal y de escenario.
    - year_label
    - escenario

- Identificación del proveedor. 
    - provider_id
    - proveedor
    - pais
    - ccaa
    - tipo_proveedor

- Atributos operativos.
    - lead_time_dias_sim
    - pedido_minimo_sim
    - n_productos

- Métricas agregadas.
    - units_value
    - sales_value

- Métricas analíticas. 
    - ranking_proveedor
    - pct_contribucion
    - pct_contribucion_acumulada

🟪 **4. Lógica utilizada.**

La vista parte de vw_ventas_base, donde cada fila representa un producto por semana.

Para calcular el ranking de proveedores, se agregan las métricas a nivel proveedor, año y escenario.

Para 2024 y 2025 se utilizan las ventas reales:

- SUM(sales_units)
- SUM(sales_value_estimated)

Para 2026 se utilizan las métricas de forecast:

- SUM(forecast_base_units)
- SUM(forecast_base_value_estimated)

- SUM(forecast_optimista_units)
- SUM(forecast_optimista_value_estimated)

- SUM(forecast_pesimista_units)
- SUM(forecast_pesimista_value_estimated)

Al igual que en el ranking de productos, se crea la columna escenario para unificar ventas reales y forecast dentro de una misma estructura.

🟪 **5. Uso de UNION ALL.**

Se utiliza UNION ALL para apilar los distintos bloques:

Ventas reales 2024-2025
+
Forecast base 2026
+
Forecast optimista 2026
+
Forecast pesimista 2026

Este enfoque evita crear vistas separadas para cada escenario y permite explotar todos los escenarios desde una única fuente.

🟪 **6. Uso de PARTITION BY.**

El ranking se calcula con:

```yaml
RANK() OVER (
    PARTITION BY year_label, escenario
    ORDER BY sales_value DESC
) AS ranking_proveedor

Esto indica que el ranking se reinicia para cada combinación de año y escenario.

Por tanto, Snowflake calcula rankings independientes para:

- 2024 - Real
- 2025 - Real
- 2026 - Forecast base
- 2026 - Forecast optimista
- 2026 - Forecast pesimista


🟪 **7. Contribución individual.**

La contribución individual indica qué porcentaje representa cada proveedor sobre el total del año y escenario correspondiente.

Se calcula dividiendo el valor del proveedor entre el total de ventas o forecast de ese mismo año y escenario:


```yaml
sales_value
/
SUM(sales_value) OVER (
    PARTITION BY year_label, escenario
)

Se utiliza NULLIF(..., 0) para evitar divisiones entre cero.

🟪 **8. Contribución acumulada.**

La contribución acumulada permite saber cuánto peso acumulan los proveedores siguiendo el orden del ranking.

Esto es útil para detectar concentración de ventas en pocos proveedores.

Por ejemplo, 

Los 3 principales proveedores concentran el 65% del valor total.

Este tipo de análisis puede ayudar a detectar dependencia de proveedores concretos.

🟪 **9. Decisión técnica.**

La vista se crea en Snowflake porque el ranking anual de proveedores por escenario es una lógica reutilizable.

Además, Looker Studio tiene más limitaciones que Power BI para construir rankings complejos y comparativas entre escenarios. Por eso se prepara esta capa analítica previamente en Snowflake.

🟪 **10. Limitaciones.**

El ranking generado es fijo o precalculado.

Esto significa que no se recalcula dinámicamente si en Power BI o Looker Studio se aplican filtros adicionales no contemplados en la partición.

Por ejemplo, si el usuario filtra por una categoría concreta, el ranking seguirá mostrando la posición global del proveedor dentro del año y escenario, no necesariamente su posición dentro de esa categoría.

Para rankings completamente dinámicos se utilizarán medidas DAX en Power BI.

🟪 **11. Uso posterior.**

Esta vista se utilizará para:

- ranking anual de proveedores, 
- análisis de concentración por proveedor, 
- comparación de proveedores entre años, 
- comparación de proveedores según escenarios para 2026, 
- identificación de proveedores clave, 
- posibles visualizaciones de dependencia o riesgo operativo. 

Además, servirá como base para constuir vw_comparativa_ranking_proveedores_anual

##### Matiz importante

Esta vista no solo sirve para “top proveedores”. También te ayuda a justificar decisiones de negocio tipo:

```text
¿Dependemos demasiado de pocos proveedores?
¿El forecast 2026 aumenta la concentración?
¿Qué proveedores ganan peso en el escenario optimista?
¿Qué proveedores pierden relevancia en el escenario pesimista?

#### ➡️ **Vista 4 - vw_comparativa_ranking_productos_anual.**

Esta vista parte de la anteriormente creada: vw_ranking_productos_anual_escenario

Sirve para comparar:

- 2024 Real → 2025 Real
- 2025 Real → 2026 Forecast base
- 2025 Real → 2026 Forecast optimista
- 2025 Real → 2026 Forecast pesimista

```yaml
CREATE OR REPLACE VIEW vw_comparativa_ranking_productos_anual AS

WITH ranking_base AS (
    SELECT *
    FROM vw_ranking_productos_anual_escenario
),

comparativa AS (
    SELECT
        actual.product_id,
        actual.nombre,
        actual.marca,
        actual.categoria,
        actual.proveedor,

        anterior.year_label AS year_anterior,
        actual.year_label AS year_actual,

        anterior.escenario AS escenario_anterior,
        actual.escenario AS escenario_actual,

        anterior.units_value AS units_value_anterior,
        actual.units_value AS units_value_actual,

        anterior.sales_value AS sales_value_anterior,
        actual.sales_value AS sales_value_actual,

        anterior.ranking_producto AS ranking_anterior,
        actual.ranking_producto AS ranking_actual,

        anterior.pct_contribucion AS pct_contribucion_anterior,
        actual.pct_contribucion AS pct_contribucion_actual,

        anterior.pct_contribucion_acumulada AS pct_contribucion_acumulada_anterior,
        actual.pct_contribucion_acumulada AS pct_contribucion_acumulada_actual

    FROM ranking_base actual
    LEFT JOIN ranking_base anterior
        ON actual.product_id = anterior.product_id
        AND actual.year_label = anterior.year_label + 1
        AND anterior.escenario = 'Real'

    WHERE actual.year_label IN (2025, 2026)
)

SELECT
    *,

    units_value_actual - units_value_anterior AS var_units_abs,

    (units_value_actual - units_value_anterior)
        / NULLIF(units_value_anterior, 0) AS var_units_pct,

    sales_value_actual - sales_value_anterior AS var_value_abs,

    (sales_value_actual - sales_value_anterior)
        / NULLIF(sales_value_anterior, 0) AS var_value_pct,

    ranking_actual - ranking_anterior AS variacion_ranking,

    pct_contribucion_actual - pct_contribucion_anterior AS var_pct_contribucion,

    CASE
        WHEN ranking_anterior IS NULL 
             AND ranking_actual IS NOT NULL
            THEN 'Nuevo en ranking'

        WHEN ranking_actual < ranking_anterior
            THEN 'Sube posiciones'

        WHEN ranking_actual > ranking_anterior
            THEN 'Baja posiciones'

        WHEN ranking_actual = ranking_anterior
            THEN 'Mantiene posición'

        ELSE 'Sin clasificar'
    END AS estado_ranking,

    CAST(year_anterior AS VARCHAR)
        || ' '
        || escenario_anterior
        || ' vs '
        || CAST(year_actual AS VARCHAR)
        || ' '
        || escenario_actual AS periodo_comparativo

FROM comparativa;

🟪 **1. Qué problema resuelve.**

La vista `vw_comparativa_ranking_productos_anual` se crea para comparar la evolución del ranking de productos entre años consecutivos.

Parte de la vista `vw_ranking_productos_anual_escenario`, donde ya se ha calculado el ranking anual de productos para ventas reales y escenarios de forecast.

Esta vista permite responder preguntas como:

- qué productos suben posiciones de 2024 a 2025;
- qué productos bajan posiciones;
- qué productos mantienen su posición;
- cómo cambia el ranking previsto en 2026 según el escenario;
- qué productos ganan o pierden peso relativo;
- qué productos aumentan o reducen su contribución sobre el total.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:

1 fila = 1 producto + 1 periodo comparativo + 1 escenario actual

🟪 **3. Columnas principales generadas.**

La vista incorpora:

- Identificación de producto.
    - product_id
    - nombre
    - marca
    - categoria
    - proveedor

- Periodos comparados.
    - year_anterior
    - year_actual
    - escenario_anterior
    - escenario_actual
    - periodo_comparativo

- Métricas comparadas. 
    - units_value_anterior
    - units_value_actual
    - sales_value_anterior
    - sales_value_actual

- Rankings comparados.
    - ranking_anterior
    - ranking_actual
    - variacion_ranking
    - estado_ranking

- Contribución. 
    - pct_contribucion_anterior
    - pct_contribucion_actual
    - var_pct_contribucion

🟪 **4. Lógica utilizada.**

La vista utiliza un LEFT JOIN de la tabla de ranking contra sí misma. Este patrón se conoce como self join.

La lógica consiste en unir cada producto del año actual con el mismo producto del año anterior.

La condición principal es:

```yaml
actual.product_id = anterior.product_id
AND actual.year_label = anterior.year_label + 1

Esto permite comparar:

- 2025 con 2024
- 2026 con 2024

Además, se fuerza que el año anterior sea siempre el escenario real:

```yaml
AND anterior.escenario = 'Real'

Esto es importante porque 2025 se compara contra 2024 real, y los escenarios de 2026 se comparan contra 2025 real.

🟪 **5. Comparativas generadas.**

La vista genera automáticamente varias comparativas:

- 2024 Real vs 2025 Real
- 2025 Real vs 2026 Forecast base
- 2025 Real vs 2026 Forecast optimista
- 2025 Real vs 2026 Forecast pesimista

Esto permite analizar tanto la evolución histórica como la evolución prevista.

🟪 **6. Variación de unidades y valor.**

La variación absoluta se calcula como: *sales_value_actual - sales_value_anterior*

La variación porcentual se calcula como:

```yaml

(sales_value_actual - sales_value_anterior)
/
NULLIF(sales_value_anterior, 0)

🟪 **7. Variación de ranking.**

La variación de ranking se calcula como: *ranking_actual - ranking_anterior*

Esta métrica requiere una interpretación especial:

- Valor negativo → mejora de posición
- Valor positivo → empeora posición
- Valor 0        → mantiene posición

Aunque matemáticamente un valor negativo pueda parecer una caída, en ranking significa mejora porque el producto se acerca al puesto 1.

🟪 **8. Variación de ranking.**

Para facilitar la interpretación, se crea un campo textual llamado estado_ranking.

Este campo clasifica cada producto como:

- Sube posiciones
- Baja posiciones
- Mantiene posición
- Nuevo en ranking
- Sin clasificar

Esto permite construir visualizaciones más intuitivas en Power BI o Looker Studio.

🟪 **9. Decisión técnica.**

La comparativa se construye en Snowflake porque es una lógica reutilizable y relativamente compleja para mantener directamente en Looker Studio.

Prepararla en Snowflake permite disponer de una vista ya lista para análisis comparativo, reduciendo la complejidad en la capa de visualización.

Esta vista actúa como una tabla analítica derivada, equivalente conceptualmente a una NDT o tabla derivada en Looker.

🟪 **10. Limitaciones.**

La comparación se basa en rankings precalculados en la vista anterior.

Por tanto, la variación de ranking compara posiciones dentro del contexto definido previamente:

año + escenario

Si después se aplican filtros adicionales en Power BI o Looker Studio, el ranking no se recalcula dinámicamente.

Para rankings totalmente dinámicos, se utilizarán medidas DAX en Power BI.

🟪 **11. Uso posterior.**

Esta vista se utilizará para:

- tablas comparativas de evolución de productos;
- análisis de subida o bajada de posiciones;
- identificación de productos con mayor crecimiento;
- identificación de productos con pérdida de relevancia;
- análisis de escenarios 2026;
- visualizaciones de cambio de ranking;
- filtros por periodo_comparativo.

En Looker Studio o Power BI, el campo periodo_comparativo permitirá seleccionar fácilmente la comparación deseada:

- 2024 Real vs 2025 Real
- 2025 Real vs 2026 Forecast base
- 2025 Real vs 2026 Forecast optimista
- 2025 Real vs 2026 Forecast pesimista

#### ➡️ **Vista 5 - vw_comparativa_ranking_proveedores_anual.**

Esta vista compara la evolución del ranking de proveedores entre años y escenarios.

Parte de: vw_ranking_proveedores_anual_escenario

Permite comparar:

- 2024 Real → 2025 Real
- 2025 Real → 2026 Forecast base
- 2025 Real → 2026 Forecast optimista
- 2025 Real → 2026 Forecast pesimista

```yaml
CREATE OR REPLACE VIEW vw_comparativa_ranking_proveedores_anual AS

WITH ranking_base AS (
    SELECT *
    FROM vw_ranking_proveedores_anual_escenario
),

comparativa AS (
    SELECT
        actual.provider_id,
        actual.proveedor,
        actual.pais,
        actual.ccaa,
        actual.tipo_proveedor,
        actual.lead_time_dias_sim,
        actual.pedido_minimo_sim,

        anterior.year_label AS year_anterior,
        actual.year_label AS year_actual,

        anterior.escenario AS escenario_anterior,
        actual.escenario AS escenario_actual,

        anterior.n_productos AS n_productos_anterior,
        actual.n_productos AS n_productos_actual,

        anterior.units_value AS units_value_anterior,
        actual.units_value AS units_value_actual,

        anterior.sales_value AS sales_value_anterior,
        actual.sales_value AS sales_value_actual,

        anterior.ranking_proveedor AS ranking_anterior,
        actual.ranking_proveedor AS ranking_actual,

        anterior.pct_contribucion AS pct_contribucion_anterior,
        actual.pct_contribucion AS pct_contribucion_actual,

        anterior.pct_contribucion_acumulada AS pct_contribucion_acumulada_anterior,
        actual.pct_contribucion_acumulada AS pct_contribucion_acumulada_actual

    FROM ranking_base actual
    LEFT JOIN ranking_base anterior
        ON actual.provider_id = anterior.provider_id
        AND actual.year_label = anterior.year_label + 1
        AND anterior.escenario = 'Real'

    WHERE actual.year_label IN (2025, 2026)
)

SELECT
    *,

    n_productos_actual - n_productos_anterior AS var_n_productos_abs,

    units_value_actual - units_value_anterior AS var_units_abs,

    (units_value_actual - units_value_anterior)
        / NULLIF(units_value_anterior, 0) AS var_units_pct,

    sales_value_actual - sales_value_anterior AS var_value_abs,

    (sales_value_actual - sales_value_anterior)
        / NULLIF(sales_value_anterior, 0) AS var_value_pct,

    ranking_actual - ranking_anterior AS variacion_ranking,

    pct_contribucion_actual - pct_contribucion_anterior AS var_pct_contribucion,

    CASE
        WHEN ranking_anterior IS NULL 
             AND ranking_actual IS NOT NULL
            THEN 'Nuevo en ranking'

        WHEN ranking_actual < ranking_anterior
            THEN 'Sube posiciones'

        WHEN ranking_actual > ranking_anterior
            THEN 'Baja posiciones'

        WHEN ranking_actual = ranking_anterior
            THEN 'Mantiene posición'

        ELSE 'Sin clasificar'
    END AS estado_ranking,

    CAST(year_anterior AS VARCHAR)
        || ' '
        || escenario_anterior
        || ' vs '
        || CAST(year_actual AS VARCHAR)
        || ' '
        || escenario_actual AS periodo_comparativo

FROM comparativa;

🟪 **1. Qué problema resuelve.**

La vista `vw_comparativa_ranking_proveedores_anual` se crea para comparar la evolución del ranking de proveedores entre años consecutivos y escenarios de forecast.

Parte de la vista `vw_ranking_proveedores_anual_escenario`, donde ya se ha calculado el ranking anual de proveedores para ventas reales y previsiones.

Esta vista permite responder preguntas como:

- qué proveedores suben posiciones de 2024 a 2025;
- qué proveedores pierden relevancia;
- qué proveedores mantienen una posición estable;
- qué proveedores ganan peso previsto en 2026;
- si el escenario optimista o pesimista cambia la importancia relativa de los proveedores;
- si aumenta o disminuye la dependencia de ciertos proveedores.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:


1 fila = 1 proveedor + 1 periodo comparativo + 1 escenario actual

🟪 **3. Columnas principales generadas.**

La vista incorpora:

- Identificación del proveedor
    - provider_id
    - proveedor
    - pais
    - ccaa
    - tipo_proveedor
- Atributos operativos
    - lead_time_dias_sim
    - pedido_minimo_sim
- Periodos comparados
    - year_anterior
    - year_actual
    - escenario_anterior
    - escenario_actual
    - periodo_comparativo
- Métricas comparadas
    - n_productos_anterior
    - n_productos_actual
    - units_value_anterior
    - units_value_actual
    - sales_value_anterior
    - sales_value_actual
- Rankings comparados
    - ranking_anterior
    - ranking_actual
    - variacion_ranking
    - estado_ranking
- Contribución
    - pct_contribucion_anterior
    - pct_contribucion_actual
    - var_pct_contribucion

🟪 **4. Lógica utilizada.**

La vista utiliza un LEFT JOIN de la tabla de ranking de proveedores contra sí misma. Este patrón se conoce como self join.

La lógica consiste en unir cada proveedor del año actual con ese mismo proveedor en el año anterior.

La condición principal es:

actual.provider_id = anterior.provider_id
AND actual.year_label = anterior.year_label + 1

Esto permite comparar:

- 2025 con 2024
- 2026 con 2025

Además, se fuerza que el año anterior sea siempre el escenario real: AND anterior.escenario = 'Real'

Esto es importante porque los escenarios de 2026 se comparan contra 2025 real, no contra otro forecast.


🟪 **5. Comparativas generadas.**

La vista genera automáticamente estas comparativas:

- 2024 Real vs 2025 Real
- 2025 Real vs 2026 Forecast base
- 2025 Real vs 2026 Forecast optimista
- 2025 Real vs 2026 Forecast pesimista

Esto permite analizar tanto la evolución histórica como la evolución prevista por escenario.

🟪 **6. Variación de productos asociados.**

Además de comparar ventas y ranking, esta vista incluye: n_productos_actual - n_productos_anterior AS var_n_productos_abs

Esta métrica permite ver si un proveedor gana o pierde presencia dentro del catálogo analizado.

Por ejemplo, un proveedor puede subir en ventas no solo porque venda más por producto, sino porque tenga más productos asociados dentro del modelo.


🟪 **7. Variación de unidades y valor.**

La variación absoluta se calcula como: sales_value_actual - sales_value_anterior

La variación porcentual se calcula como:

```yaml
(sales_value_actual - sales_value_anterior)
/
NULLIF(sales_value_anterior, 0)

🟪 **8. Variación de ranking.**

La variación de ranking se calcula como: ranking_actual - ranking_anterior

Esta métrica requiere una interpretación específica:

- Valor negativo → mejora de posición
- Valor positivo → empeora posición
- Valor 0        → mantiene posición

🟪 **9. Estado del ranking.**

Para facilitar la interpretación, se crea un campo textual llamado estado_ranking.

Este campo clasifica cada proveedor como:

- Sube posiciones
- Baja posiciones
- Mantiene posición
- Nuevo en ranking
- Sin clasificar

Esto permite crear visualizaciones más claras y segmentar proveedores según su evolución.

🟪 **10. Decisión técnica.**

La comparativa de ranking de proveedores se construye en Snowflake porque es una lógica reutilizable, estable y más compleja de mantener directamente en Looker Studio.

Al dejarla preparada en Snowflake, la capa de visualización puede centrarse en consumir la información y representarla de forma clara.

Esta vista funciona como una tabla analítica derivada, equivalente conceptualmente a una NDT o tabla derivada en Looker.

🟪 **11. Limitaciones.**

La comparación se basa en rankings precalculados en la vista anterior.

Por tanto, la variación de ranking compara posiciones dentro del contexto definido previamente:

año + escenario

Si después se aplican filtros adicionales en Power BI o Looker Studio, el ranking no se recalcula dinámicamente.

Para rankings completamente dinámicos se utilizarán medidas DAX en Power BI.

🟪 **12. Uso posterior.**

Esta vista se utilizará para:

- análisis de evolución de proveedores;
- detección de proveedores que ganan peso;
- detección de proveedores que pierden relevancia;
- análisis de dependencia por proveedor;
- comparación de escenarios 2026;
- visualizaciones de cambio de ranking;
- filtros por periodo_comparativo.

En Power BI o Looker Studio, el campo periodo_comparativo permitirá seleccionar comparativas como:

- 2024 Real vs 2025 Real
- 2025 Real vs 2026 Forecast base
- 2025 Real vs 2026 Forecast optimista
- 2025 Real vs 2026 Forecast pesimista


#### Idea clave de esta vista

Esta vista no recalcula el ranking desde cero.

Hace esto:

1. Toma el ranking anual de proveedores ya calculado
2. Une cada proveedor con su año anterior
3. Calcula variaciones de ventas, contribución y posición
4. Clasifica si sube, baja o se mantiene

#### ➡️ **Vista 6 - vw_ventas_yoy_producto_semana.**

Esta vista compara cada producto contra el mismo producto en la misma semana del año anterior.

Genera comparativas como:

- 2024 Real → 2025 Real
- 2025 Real → 2026 Forecast base
- 2025 Real → 2026 Forecast optimista
- 2025 Real → 2026 Forecast pesimista

```yaml
CREATE OR REPLACE VIEW vw_ventas_yoy_producto_semana AS

WITH metricas_producto_semana AS (

    -- Ventas reales 2024 y 2025
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Real' AS escenario,

        product_id,
        nombre,
        marca,
        categoria,
        proveedor,

        sales_units AS units_value,
        sales_value_estimated AS sales_value,

        sales_units_adjusted AS units_value_adjusted,
        sales_value_estimated_adjusted AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label IN (2024, 2025)

    UNION ALL

    -- Forecast 2026: escenario base
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast base' AS escenario,

        product_id,
        nombre,
        marca,
        categoria,
        proveedor,

        forecast_base_units AS units_value,
        forecast_base_value_estimated AS sales_value,

        forecast_base_units_adjusted AS units_value_adjusted,
        forecast_base_value_estimated_adjusted AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026

    UNION ALL

    -- Forecast 2026: escenario optimista
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast optimista' AS escenario,

        product_id,
        nombre,
        marca,
        categoria,
        proveedor,

        forecast_optimista_units AS units_value,
        forecast_optimista_value_estimated AS sales_value,

        forecast_optimista_units_adjusted AS units_value_adjusted,
        forecast_optimista_value_estimated_adjusted AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026

    UNION ALL

    -- Forecast 2026: escenario pesimista
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast pesimista' AS escenario,

        product_id,
        nombre,
        marca,
        categoria,
        proveedor,

        forecast_pesimista_units AS units_value,
        forecast_pesimista_value_estimated AS sales_value,

        forecast_pesimista_units_adjusted AS units_value_adjusted,
        forecast_pesimista_value_estimated_adjusted AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026
),

comparativa AS (
    SELECT
        actual.product_id,
        actual.nombre,
        actual.marca,
        actual.categoria,
        actual.proveedor,

        anterior.year_label AS year_anterior,
        actual.year_label AS year_actual,

        anterior.escenario AS escenario_anterior,
        actual.escenario AS escenario_actual,

        actual.week_number_business,
        anterior.year_week_key AS year_week_key_anterior,
        actual.year_week_key AS year_week_key_actual,

        anterior.week_label AS week_label_anterior,
        actual.week_label AS week_label_actual,

        actual.week_start_date,
        actual.week_end_date,

        actual.is_partial_week,
        actual.is_campaign_week,
        actual.campaign_name_primary,
        actual.campaign_group_primary,

        anterior.units_value AS units_value_anterior,
        actual.units_value AS units_value_actual,

        anterior.sales_value AS sales_value_anterior,
        actual.sales_value AS sales_value_actual,

        anterior.units_value_adjusted AS units_value_adjusted_anterior,
        actual.units_value_adjusted AS units_value_adjusted_actual,

        anterior.sales_value_adjusted AS sales_value_adjusted_anterior,
        actual.sales_value_adjusted AS sales_value_adjusted_actual

    FROM metricas_producto_semana actual
    LEFT JOIN metricas_producto_semana anterior
        ON actual.product_id = anterior.product_id
        AND actual.week_number_business = anterior.week_number_business
        AND actual.year_label = anterior.year_label + 1
        AND anterior.escenario = 'Real'

    WHERE actual.year_label IN (2025, 2026)
)

SELECT
    *,

    units_value_actual - units_value_anterior AS var_units_abs,

    (units_value_actual - units_value_anterior)
        / NULLIF(units_value_anterior, 0) AS var_units_pct,

    sales_value_actual - sales_value_anterior AS var_value_abs,

    (sales_value_actual - sales_value_anterior)
        / NULLIF(sales_value_anterior, 0) AS var_value_pct,

    units_value_adjusted_actual - units_value_adjusted_anterior 
        AS var_units_adjusted_abs,

    (units_value_adjusted_actual - units_value_adjusted_anterior)
        / NULLIF(units_value_adjusted_anterior, 0) AS var_units_adjusted_pct,

    sales_value_adjusted_actual - sales_value_adjusted_anterior 
        AS var_value_adjusted_abs,

    (sales_value_adjusted_actual - sales_value_adjusted_anterior)
        / NULLIF(sales_value_adjusted_anterior, 0) AS var_value_adjusted_pct,

    CAST(year_anterior AS VARCHAR)
        || ' '
        || escenario_anterior
        || ' vs '
        || CAST(year_actual AS VARCHAR)
        || ' '
        || escenario_actual AS periodo_comparativo

FROM comparativa;


🟪 **1. Qué problema resuelve.**

La vista `vw_ventas_yoy_producto_semana` se crea para analizar la evolución interanual de las ventas a nivel de producto y semana.

El objetivo es comparar cada producto con su comportamiento en la misma semana del año anterior.

Esta vista permite responder preguntas como:

- cómo evolucionan las ventas semanales de un producto de 2024 a 2025;
- qué productos crecen o caen semana a semana;
- cómo se comporta el forecast 2026 frente a las ventas reales de 2025;
- qué diferencias existen entre los escenarios base, optimista y pesimista;
- qué semanas presentan mayor variación interanual;
- cómo afectan campañas como Black Friday, Navidad o rebajas a la evolución semanal.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:

1 fila = 1 producto + 1 semana + 1 periodo comparativo + 1 escenario actual

🟪 **3. Columnas principales generadas.**

- Identificación del producto
    - product_id
    - nombre
    - marca
    - categoria
    - proveedor
- Información temporal
    - year_anterior
    - year_actual
    - week_number_business
    - year_week_key_anterior
    - year_week_key_actual
    - week_label_anterior
    - week_label_actual
    - week_start_date
    - week_end_date
- Escenarios
    - escenario_anterior
    - escenario_actual
    - periodo_comparativo
- Contexto comercial
    - is_partial_week
    - is_campaign_week
    - campaign_name_primary
    - campaign_group_primary
- Métricas comparadas
    - units_value_anterior
    - units_value_actual
    - sales_value_anterior
    - sales_value_actual
- Métricas ajustadas
    - units_value_adjusted_anterior
    - units_value_adjusted_actual
    - sales_value_adjusted_anterior
    - sales_value_adjusted_actual
- Variaciones
    - var_units_abs
    - var_units_pct
    - var_value_abs
    - var_value_pct
    - var_units_adjusted_abs
    - var_units_adjusted_pct
    - var_value_adjusted_abs
    - var_value_adjusted_pct

🟪 **4. Lógica utilizada.**

La vista parte de vw_ventas_base.

El primer bloque, metricas_producto_semana, transforma las ventas reales y las previsiones de 2026 en una estructura común.

Para 2024 y 2025 se utilizan las ventas reales:

- sales_units
- sales_value_estimated
- sales_units_adjusted
- sales_value_estimated_adjusted

Para 2026 se utilizan las métricas de forecast en tres escenarios:

- forecast_base_units
- forecast_base_value_estimated

- forecast_optimista_units
- forecast_optimista_value_estimated

- forecast_pesimista_units
- forecast_pesimista_value_estimated

La columna escenario permite unificar ventas reales y previsiones dentro de una misma estructura.

🟪 **5. Uso de UNION ALL.**

Se utiliza UNION ALL para apilar cuatro bloques:

Ventas reales 2024-2025
+
Forecast base 2026
+
Forecast optimista 2026
+
Forecast pesimista 2026


Este enfoque permite comparar años reales y escenarios de forecast sin crear una vista distinta para cada escenario.

La estructura resultante permite trabajar siempre con las mismas columnas:

year_label
week_number_business
escenario
product_id
units_value
sales_value

🟪 **6. Uso del self join para comparar años.**

La comparación YoY se realiza mediante un LEFT JOIN de la tabla contra sí misma.

La parte principal es:

```yaml
actual.product_id = anterior.product_id
AND actual.week_number_business = anterior.week_number_business
AND actual.year_label = anterior.year_label + 1
AND anterior.escenario = 'Real'

Esto significa:

- se compara el mismo producto;
- se compara la misma semana de negocio;
- se compara el año actual contra el año anterior;
- el año anterior debe corresponder siempre a ventas reales.

Por tanto, la vista compara:

- 2025 Real contra 2024 Real
- 2026 Forecast base contra 2025 Real
- 2026 Forecast optimista contra 2025 Real
- 2026 Forecast pesimista contra 2025 Real


🟪 **7. Por qué se usa week_number_business.**

Se utiliza week_number_business para alinear semanas equivalentes entre años.

Por ejemplo:

Semana 48 de 2025 → Semana 48 de 2024
Semana 49 de 2026 → Semana 49 de 2025

Esto permite analizar evolución interanual a nivel semanal.

No obstante, hay que tener cuidado con las semanas parciales al inicio y al final del año, ya que pueden tener menos días y distorsionar la comparación visual.

Por este motivo, la vista conserva tanto las métricas originales como las métricas ajustadas.

🟪 **8. Métricas ajustadas y semanas parciales.**

Las semanas parciales pueden distorsionar la comparación porque no todas representan siete días completos.

Por ejemplo:

2025_W01 puede tener 5 días
2026_W01 puede tener 4 días

Si se comparan unidades reales sin ajustar, una semana con menos días podría parecer artificialmente peor.

Por eso la vista incluye métricas ajustadas:

units_value_adjusted
sales_value_adjusted

Estas métricas permiten realizar comparaciones más estables cuando existen semanas parciales.

En visualizaciones semanales, conviene indicar si se están utilizando valores reales o valores ajustados.

🟪 **9. Variaciones absolutas y porcentuales.**

La variación absoluta mide la diferencia directa entre el periodo actual y el anterior: sales_value_actual - sales_value_anterior

La variación porcentual mide esa diferencia en relación con el periodo anterior:

```yaml
(sales_value_actual - sales_value_anterior)
/
NULLIF(sales_value_anterior, 0)

🟪 **10. Decisión técnica.**

La vista se construye en Snowflake porque la lógica YoY semanal es reutilizable y puede ser compleja de recrear directamente en Looker Studio.

Además, al preparar esta vista en Snowflake, se facilita que Power BI y Looker Studio consuman una estructura ya preparada para análisis interanual.

Esta vista funciona como una tabla analítica derivada sobre el modelo estrella.

🟪 **11. Limitaciones.**

La comparación se basa en la alineación por número de semana de negocio.

Esto es útil para análisis YoY, pero no siempre garantiza una equivalencia exacta de calendario, especialmente cuando existen:

- semanas parciales;
- años con distinta distribución de días;
- campañas que pueden caer en semanas diferentes;
- festividades móviles, como Semana Santa.

Por tanto, las visualizaciones deben interpretarse teniendo en cuenta el contexto temporal y comercial.

🟪 **12. Uso posterior.**

Esta vista se utilizará para:

- gráficos de evolución semanal por producto;
- análisis YoY de ventas reales 2025 vs 2024;
- comparación de forecast 2026 contra ventas reales 2025;
- análisis de escenarios;
- detección de semanas con crecimiento o caída relevante;
- visualizaciones de campañas;
- análisis de productos concretos.

#### ➡️ **Vista 7 - vw_ventas_yoy_categoria_semana.**

Esta vista es muy parecida a la anterior, pero cambia el nivel de análisis. El producto + semana pasa a ser categoría + semana. 

Es decir, en vez de ver cómo evoluciona cada producto, analizamos cómo evoluciona cada categoría semana a semana respecto al año anterior.

```yaml
CREATE OR REPLACE VIEW vw_ventas_yoy_categoria_semana AS

WITH metricas_categoria_semana AS (

    -- Ventas reales 2024 y 2025
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Real' AS escenario,

        category_id,
        categoria,
        familia_categoria,
        formato_categoria,

        SUM(sales_units) AS units_value,
        SUM(sales_value_estimated) AS sales_value,

        SUM(sales_units_adjusted) AS units_value_adjusted,
        SUM(sales_value_estimated_adjusted) AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label IN (2024, 2025)
    GROUP BY
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,
        category_id,
        categoria,
        familia_categoria,
        formato_categoria

    UNION ALL

    -- Forecast 2026: escenario base
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast base' AS escenario,

        category_id,
        categoria,
        familia_categoria,
        formato_categoria,

        SUM(forecast_base_units) AS units_value,
        SUM(forecast_base_value_estimated) AS sales_value,

        SUM(forecast_base_units_adjusted) AS units_value_adjusted,
        SUM(forecast_base_value_estimated_adjusted) AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,
        category_id,
        categoria,
        familia_categoria,
        formato_categoria

    UNION ALL

    -- Forecast 2026: escenario optimista
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast optimista' AS escenario,

        category_id,
        categoria,
        familia_categoria,
        formato_categoria,

        SUM(forecast_optimista_units) AS units_value,
        SUM(forecast_optimista_value_estimated) AS sales_value,

        SUM(forecast_optimista_units_adjusted) AS units_value_adjusted,
        SUM(forecast_optimista_value_estimated_adjusted) AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,
        category_id,
        categoria,
        familia_categoria,
        formato_categoria

    UNION ALL

    -- Forecast 2026: escenario pesimista
    SELECT
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,

        'Forecast pesimista' AS escenario,

        category_id,
        categoria,
        familia_categoria,
        formato_categoria,

        SUM(forecast_pesimista_units) AS units_value,
        SUM(forecast_pesimista_value_estimated) AS sales_value,

        SUM(forecast_pesimista_units_adjusted) AS units_value_adjusted,
        SUM(forecast_pesimista_value_estimated_adjusted) AS sales_value_adjusted

    FROM vw_ventas_base
    WHERE year_label = 2026
    GROUP BY
        year_label,
        week_number_business,
        year_week_key,
        week_label,
        week_start_date,
        week_end_date,
        is_partial_week,
        is_campaign_week,
        campaign_name_primary,
        campaign_group_primary,
        category_id,
        categoria,
        familia_categoria,
        formato_categoria
),

comparativa AS (
    SELECT
        actual.category_id,
        actual.categoria,
        actual.familia_categoria,
        actual.formato_categoria,

        anterior.year_label AS year_anterior,
        actual.year_label AS year_actual,

        anterior.escenario AS escenario_anterior,
        actual.escenario AS escenario_actual,

        actual.week_number_business,
        anterior.year_week_key AS year_week_key_anterior,
        actual.year_week_key AS year_week_key_actual,

        anterior.week_label AS week_label_anterior,
        actual.week_label AS week_label_actual,

        actual.week_start_date,
        actual.week_end_date,

        actual.is_partial_week,
        actual.is_campaign_week,
        actual.campaign_name_primary,
        actual.campaign_group_primary,

        anterior.units_value AS units_value_anterior,
        actual.units_value AS units_value_actual,

        anterior.sales_value AS sales_value_anterior,
        actual.sales_value AS sales_value_actual,

        anterior.units_value_adjusted AS units_value_adjusted_anterior,
        actual.units_value_adjusted AS units_value_adjusted_actual,

        anterior.sales_value_adjusted AS sales_value_adjusted_anterior,
        actual.sales_value_adjusted AS sales_value_adjusted_actual

    FROM metricas_categoria_semana actual
    LEFT JOIN metricas_categoria_semana anterior
        ON actual.category_id = anterior.category_id
        AND actual.week_number_business = anterior.week_number_business
        AND actual.year_label = anterior.year_label + 1
        AND anterior.escenario = 'Real'

    WHERE actual.year_label IN (2025, 2026)
)

SELECT
    *,

    units_value_actual - units_value_anterior AS var_units_abs,

    (units_value_actual - units_value_anterior)
        / NULLIF(units_value_anterior, 0) AS var_units_pct,

    sales_value_actual - sales_value_anterior AS var_value_abs,

    (sales_value_actual - sales_value_anterior)
        / NULLIF(sales_value_anterior, 0) AS var_value_pct,

    units_value_adjusted_actual - units_value_adjusted_anterior 
        AS var_units_adjusted_abs,

    (units_value_adjusted_actual - units_value_adjusted_anterior)
        / NULLIF(units_value_adjusted_anterior, 0) AS var_units_adjusted_pct,

    sales_value_adjusted_actual - sales_value_adjusted_anterior 
        AS var_value_adjusted_abs,

    (sales_value_adjusted_actual - sales_value_adjusted_anterior)
        / NULLIF(sales_value_adjusted_anterior, 0) AS var_value_adjusted_pct,

    CAST(year_anterior AS VARCHAR)
        || ' '
        || escenario_anterior
        || ' vs '
        || CAST(year_actual AS VARCHAR)
        || ' '
        || escenario_actual AS periodo_comparativo

FROM comparativa;

🟪 **1. Qué problema resuelve.**

La vista `vw_ventas_yoy_categoria_semana` se crea para analizar la evolución interanual de las ventas a nivel de categoría y semana.

Mientras que la vista anterior analizaba la evolución producto a producto, esta vista permite observar patrones agregados por categoría.

Esto ayuda a responder preguntas como:

- qué categorías crecen o caen respecto al año anterior;
- qué categorías presentan mejor evolución semanal;
- cómo se comportan las categorías principales en campañas;
- cómo evoluciona el forecast 2026 frente a las ventas reales de 2025;
- qué categorías son más sensibles a los distintos escenarios de previsión;
- si el crecimiento está concentrado en productos concretos o en una categoría completa.

🟪 **2. Grano del resultado.**

La vista tiene granularidad:

1 fila = 1 categoría + 1 semana + 1 periodo comparativo + 1 escenario actual

🟪 **3. Columnas principales generadas.**

La vista incorpora:

- Identificación de categoría
    - category_id
    - categoria
    - familia_categoria
    - formato_categoria
- Información temporal
    - year_anterior
    - year_actual
    - week_number_business
    - year_week_key_anterior
    - year_week_key_actual
    - week_label_anterior
    - week_label_actual
    - week_start_date
    - week_end_date
- Escenarios
    - escenario_anterior
    - escenario_actual
    - periodo_comparativo
- Contexto comercial
    - is_partial_week
    - is_campaign_week
    - campaign_name_primary
    - campaign_group_primary
- Métricas comparadas
    - units_value_anterior
    - units_value_actual
    - sales_value_anterior
    - sales_value_actual
- Métricas ajustadas
    - units_value_adjusted_anterior
    - units_value_adjusted_actual
    - sales_value_adjusted_anterior
    - sales_value_adjusted_actual
- Variaciones
    - var_units_abs
    - var_units_pct
    - var_value_abs
    - var_value_pct
    - var_units_adjusted_abs
    - var_units_adjusted_pct
    - var_value_adjusted_abs
    - var_value_adjusted_pct

🟪 **4. Lógica utilizada.**

La vista parte de vw_ventas_base.

El primer bloque, metricas_categoria_semana, agrega la información a nivel de categoría, año, semana y escenario.

Para 2024 y 2025 se utilizan las ventas reales:

- SUM(sales_units)
- SUM(sales_value_estimated)
- SUM(sales_units_adjusted)
- SUM(sales_value_estimated_adjusted)

Para 2026 se utilizan las previsiones en tres escenarios:

- SUM(forecast_base_units)
- SUM(forecast_base_value_estimated)

- SUM(forecast_optimista_units)
- SUM(forecast_optimista_value_estimated)

- SUM(forecast_pesimista_units)
- SUM(forecast_pesimista_value_estimated)

La columna escenario permite transformar las ventas reales y los forecasts en una estructura común.

🟪 **5. Decisión técnica.**

Se utiliza UNION ALL para apilar:

Ventas reales 2024-2025
+
Forecast base 2026
+
Forecast optimista 2026
+
Forecast pesimista 2026

Esto permite analizar todos los escenarios con la misma estructura y evita crear varias vistas separadas.

El resultado es una tabla lógica con columnas comunes:

- year_label
- week_number_business
- escenario
- category_id
- units_value
- sales_value

🟪 **6. Agregación por categoría.**

A diferencia de la vista vw_ventas_yoy_producto_semana, aquí las métricas se agregan con SUM().

Esto es necesario porque una categoría contiene varios productos.

Ejemplo:

Categoría Proteínas = suma de todos los productos de Proteínas

Por tanto, antes de comparar años, se calcula el total semanal de cada categoría.

🟪 **7. Uso del self join para comparar años.**

La comparación YoY se realiza mediante un LEFT JOIN de la tabla contra sí misma.

La condición principal es:

```yaml
actual.category_id = anterior.category_id
AND actual.week_number_business = anterior.week_number_business
AND actual.year_label = anterior.year_label + 1
AND anterior.escenario = 'Real'

Esto significa:

- se compara la misma categoría;
- se compara la misma semana de negocio;
- se compara el año actual contra el año anterior;
- el año anterior siempre corresponde a ventas reales.

Por tanto, la vista compara:

- 2025 Real contra 2024 Real
- 2026 Forecast base contra 2025 Real
- 2026 Forecast optimista contra 2025 Real
- 2026 Forecast pesimista contra 2025 Real

🟪 **8. Por qué se usa week_number_business.**

Se utiliza week_number_business para alinear semanas equivalentes entre años.

Por ejemplo:

Semana 48 de 2025 → Semana 48 de 2024
Semana 49 de 2026 → Semana 49 de 2025

Esto permite comparar la evolución semanal de cada categoría de forma homogénea.

Aun así, esta comparación debe interpretarse con cuidado si existen semanas parciales o campañas que cambian ligeramente de semana entre años.

🟪 **9. Métricas ajustadas y semanas parciales.**

Las semanas parciales pueden distorsionar el análisis porque no todas contienen siete días.

Por ejemplo:

2025_W01 puede tener 5 días
2026_W01 puede tener 4 días

Por ese motivo, la vista conserva métricas originales y ajustadas.

Las métricas ajustadas permiten comparar mejor semanas parciales, especialmente al inicio y final de año.

Esto es importante en visualizaciones semanales porque una caída en una semana parcial podría deberse simplemente a que esa semana contiene menos días, no a un peor comportamiento real de la categoría.

🟪 **10. Variaciones absolutas y porcentuales.**

La variación absoluta mide la diferencia directa entre el periodo actual y el anterior:

sales_value_actual - sales_value_anterior

La variación porcentual mide el cambio relativo frente al periodo anterior:

```yaml
(sales_value_actual - sales_value_anterior)
/
NULLIF(sales_value_anterior, 0)

🟪 **11. Decisión técnica.**

La vista se construye en Snowflake porque la lógica YoY semanal por categoría es reutilizable y puede ser compleja de mantener directamente en Looker Studio.

Además, trabajar con una vista agregada por categoría reduce la complejidad para la capa de visualización y permite crear gráficos comparativos más claros.

Esta vista funciona como una tabla analítica derivada sobre el modelo estrella.

🟪 **12. Limitaciones.**

La comparación se basa en el número de semana de negocio.

Esto permite una comparación ordenada y sencilla, pero puede tener limitaciones cuando:

- existen semanas parciales;
- una campaña cae en semanas distintas según el año;
- hay festividades móviles;
- se comparan semanas con distinto número de días;
- una categoría tiene pocos productos y puede ser sensible a valores extremos.

Por ello, las métricas ajustadas y los flags de campaña deben utilizarse como contexto interpretativo.

🟪 **13. Uso posterior.**

Esta vista se utilizará para:

- gráficos de evolución semanal por categoría;
- análisis YoY de categorías;
- comparación de ventas reales 2025 frente a 2024;
- comparación de forecast 2026 frente a ventas reales 2025;
- análisis de escenarios;
- detección de categorías con mayor crecimiento o caída;
- análisis de campañas por categoría;
- visualizaciones agregadas más estables que las de producto individual.

En Power BI o Looker Studio, el campo periodo_comparativo permitirá filtrar comparativas como:

- 2024 Real vs 2025 Real
- 2025 Real vs 2026 Forecast base
- 2025 Real vs 2026 Forecast optimista
- 2025 Real vs 2026 Forecast pesimista

> Las vistas analíticas se han creado correctamente en Snowflake y se han versionado en el repositorio mediante el archivo `sql/snowflake/10_create_analytics_views.sql`, para garantizar la reproducibilidad del modelo.

### 📌 **Migración de la capa analítica a BigQuery**

## Migración de la capa analítica a BigQuery

Tras validar el modelo en Snowflake, se replica la capa analítica en BigQuery para utilizarla como fuente de datos principal en Looker Studio.

BigQuery se utiliza como alternativa más integrada con el ecosistema Google y como fuente para la versión publicable del dashboard.

Se crean dos datasets:

- `corepulse_dw`: tablas base del modelo.
- `corepulse_analytics`: vistas analíticas de consumo.

Las vistas creadas reproducen la lógica diseñada previamente en Snowflake, adaptando la sintaxis a BigQuery mediante nombres completos de tabla, `STRING` en lugar de `VARCHAR` y `SAFE_DIVIDE` para evitar errores por división entre cero.

----

## **5. Diseño y desarrollo del cuadro de mandos.**

### **5.1. Desarrollo de la página resumen en Data Studio.**

#### 🎯 **Objetivo**

La primera página del dashboard se diseña como una vista ejecutiva del negocio, orientada a resumir el comportamiento comercial del año 2025 frente al año anterior.

El objetivo principal es responder de forma rápida a preguntas como:

- cuál ha sido el volumen total de ventas TY;
- cómo evolucionan las ventas frente al año anterior;
- qué categoría concentra mayor facturación;
- qué producto lidera las ventas;
- qué proveedores tienen mayor peso comercial;
- cómo se distribuyen las ventas según una clasificación estratégica de productos.

Para mantener la coherencia de los filtros en Looker Studio, se decide utilizar como fuente principal la vista `vw_ventas_yoy_producto_semana`, ya que contiene tanto las métricas actuales como las métricas del año anterior, además de dimensiones comerciales como categoría, proveedor, marca, campaña y producto.


#### 🧮 **KPIs principales**


La página resumen incorpora los siguientes KPIs:

| KPI | Fuente | Métrica principal | Filtro aplicado |
|---|---|---|---|
| Ventas TY | `vw_ventas_yoy_producto_semana` | `SUM(sales_value_actual)` | `year_actual = 2025`, `escenario_actual = Real` |
| Variación YoY valor % | `vw_ventas_yoy_producto_semana` | `(SUM(sales_value_actual) - SUM(sales_value_anterior)) / SUM(sales_value_anterior)` | `year_actual = 2025`, `escenario_actual = Real` |
| Top categoría | `vw_ventas_yoy_producto_semana` | `SUM(sales_value_actual)` agrupado por categoría | Orden descendente y límite 1 |
| Producto top ventas | `vw_ventas_yoy_producto_semana` | `SUM(sales_value_actual)` agrupado por producto | Orden descendente y límite 1 |

La variación YoY se calcula como una métrica agregada y no como suma directa de porcentajes fila a fila, para evitar resultados incorrectos derivados de la agregación de porcentajes.

#### 📊 **Visualizaciones incluidas**

La página incorpora los siguientes bloques visuales:

➡️  **Evolución ventas TY vs LY**

Gráfico de serie temporal que compara semanalmente las ventas de 2025 frente a las ventas equivalentes de 2024.

- Dimensión temporal: `week_start_date`
- Métrica TY: `SUM(sales_value_adjusted_actual)` o `SUM(sales_value_actual)`
- Métrica LY: `SUM(sales_value_adjusted_anterior)` o `SUM(sales_value_anterior)`
- Filtros: `year_actual = 2025`, `escenario_actual = Real`

Se utiliza una fecha real como eje temporal para asegurar que Looker Studio ordene correctamente las semanas.

➡️  **Ventas TY por dimensión de análisis**

Gráfico de barras horizontales que permite alternar entre categoría, proveedor y marca mediante un parámetro de Looker Studio.

Se crea un parámetro llamado `Ver por` con los valores:

- Categoría
- Proveedor
- Marca

A partir de este parámetro se crea una dimensión calculada que devuelve el campo correspondiente según la selección del usuario.

➡️  **Clasificación estratégica de productos**

Gráfico de barras horizontales que muestra el peso de las ventas TY según la clasificación estratégica asignada a cada producto.

La clasificación se calcula previamente en BigQuery y se incorpora a la vista YoY de producto para mantener una única fuente principal en Looker Studio.

➡️  **Top proveedores por ventas TY**

Gráfico de barras horizontales que muestra los principales proveedores según ventas actuales del año 2025.

#### **5.1.1. Parámetro de dimensión dinámica**.

Para evitar duplicar gráficos por categoría, proveedor y marca, se crea un parámetro en Looker Studio que permite seleccionar la dimensión de análisis.

El parámetro se utiliza dentro de un campo calculado:

```sql
CASE
  WHEN Ver por = 'Categoría' THEN categoria
  WHEN Ver por = 'Proveedor' THEN proveedor
  WHEN Ver por = 'Marca' THEN marca
END

#### **5.1.2. Vista `vw_clasificacion_producto_estrategica`**.

- **Objetivo**

Se crea la vista `vw_clasificacion_producto_estrategica` para clasificar los productos según su relevancia comercial actual, su evolución frente al año anterior y su previsión futura.

Esta clasificación permite enriquecer el dashboard con una lectura más estratégica del catálogo, diferenciando entre productos consolidados, productos relevantes en riesgo, productos emergentes y productos rezagados.


- **Variables utilizadas**

La clasificación se calcula a nivel producto utilizando:

- ventas reales de 2025;
- ventas reales de 2024;
- variación YoY 2025 vs 2024;
- ranking de ventas 2025;
- forecast base 2026;
- variación prevista 2026 frente a 2025.

- **Reglas de clasificación**

| Clasificación | Criterio general | Interpretación |
|---|---|---|
| Líder consolidado | Producto top con comportamiento estable y señales positivas actuales o futuras | Producto fuerte y con continuidad esperada |
| Líder en riesgo | Producto top con caída histórica o previsión negativa significativa | Producto relevante que requiere seguimiento |
| Emergente | Producto no líder con crecimiento YoY y previsión positiva | Producto con potencial de desarrollo |
| Rezagado | Producto con menor peso relativo o señales débiles | Producto candidato a revisión |

Se flexibilizan las reglas de clasificación para evitar que pequeñas caídas puntuales reclasifiquen automáticamente productos relevantes como productos en riesgo.

- **Integración con `vw_ventas_yoy_producto_semana`**

Una vez creada la vista estratégica, se actualiza `vw_ventas_yoy_producto_semana` mediante un `LEFT JOIN` por `product_id`.

De esta forma, la vista YoY incorpora los campos:

- `clasificacion_producto`
- `orden_clasificacion_producto`
- `descripcion_clasificacion`
- `ranking_2025`
- `pct_contribucion_2025`
- `var_yoy_producto_pct`
- `forecast_base_value_2026`
- `var_forecast_producto_2026_pct`

Esto permite que Looker Studio siga utilizando una única fuente principal para la página resumen, manteniendo la coherencia de filtros y evitando combinaciones de datos innecesarias.

#### **5.1.3. Decisiones descartadas**.

Durante el desarrollo se valoró el uso de combinaciones de datos en Looker Studio para calcular porcentajes de contribución y rankings dinámicos.

Sin embargo, se descartó esta opción porque las combinaciones podían alterar el contexto de agregación y generar resultados inconsistentes, como porcentajes del 100% al aplicar determinados filtros.

Por este motivo, se decidió trasladar la lógica de negocio más compleja a BigQuery y utilizar Looker Studio principalmente como capa de visualización.

### **5.2. Desarrollo de la página de detalle de producto en Data Studio.**

#### 🎯 **Objetivo**

La segunda página del dashboard se diseña como una vista de análisis individual del catálogo, orientada a consultar el comportamiento de un producto concreto.

A diferencia de la página resumen, que ofrece una visión ejecutiva global del negocio, esta página permite analizar en detalle las principales señales comerciales de un producto seleccionado:

- ficha técnica del producto;
- imagen asociada al producto;
- ventas y unidades vendidas en 2025;
- variación YoY frente a 2024;
- ranking del producto dentro del catálogo;
- contribución sobre las ventas totales;
- venta media semanal;
- perfil de comportamiento;
- clasificación estratégica;
- evolución semanal TY vs LY;
- comparativa anual frente a su categoría;
- información básica del proveedor asociado.

El objetivo es que el usuario pueda entender rápidamente qué papel tiene cada producto dentro del catálogo, cómo evoluciona frente al año anterior y cómo se posiciona respecto a otros productos de su misma categoría.



#### 🧩 **Fuente principal de datos**

Para mantener la coherencia de filtros en Looker Studio, se decide crear una única vista principal para la página de detalle:

`vw_detalle_producto`

Esta vista se utiliza como fuente común para la ficha, los KPIs y las visualizaciones de la página.

La decisión se toma para evitar problemas derivados del uso de varias fuentes en Looker Studio, especialmente en páginas donde todos los elementos deben responder al mismo selector de producto.

La vista mantiene grano semanal:

`1 fila = 1 producto + 1 semana + 1 periodo comparativo`

Esto permite construir gráficos de evolución temporal, pero también incorpora campos anuales repetidos, como ranking, contribución, clasificación estratégica, forecast 2026 o venta media semanal.

#### ⚠️ **Tratamiento de métricas semanales y métricas anuales**

La vista `vw_detalle_producto` combina dos tipos de campos:

| Tipo de campo | Ejemplos | Agregación en Looker Studio |
|---|---|---|
| Métricas semanales | `sales_value_actual`, `units_value_actual`, `sales_value_anterior`, `units_value_anterior` | `SUM` |
| Métricas anuales repetidas | `ranking_2025`, `pct_contribucion_2025`, `venta_media_semanal_2025`, `forecast_base_value_2026` | `MAX` o `MIN` |
| Dimensiones descriptivas | `clasificacion_producto`, `perfil_comportamiento`, `proveedor`, `categoria` | Dimensión |

Esta distinción es importante porque los campos anuales se repiten en cada semana del producto. Por tanto, no deben sumarse en Looker Studio, ya que eso inflaría los valores.

Por ejemplo, para mostrar la contribución del producto sobre ventas TY se utiliza:

`MAX(pct_contribucion_2025)`

y no:

`SUM(pct_contribucion_2025)`

Del mismo modo, para mostrar la venta media semanal se utiliza:

`MAX(venta_media_semanal_2025)`

ya que esta métrica ya está calculada previamente a nivel producto-año.

#### 🖼️ **Integración de imágenes de producto**

Para mostrar imágenes dinámicas en la ficha de producto, se crea una dimensión auxiliar en BigQuery:

`dim_producto_imagen`

Esta tabla contiene una fila por producto y los campos necesarios para representar la imagen en Looker Studio:

| Campo | Descripción |
|---|---|
| `product_id` | Identificador del producto |
| `image_url` | URL pública de la imagen |
| `image_alt` | Texto alternativo de la imagen |

La tabla se incorpora a `vw_detalle_producto` mediante un `LEFT JOIN` por `product_id`.

En Looker Studio se crea un campo calculado utilizando la función `IMAGE()`:

```sql
IMAGE(image_url, image_alt)


#### 🧮 **KPIs principales de la página de detalle**

La página de detalle incorpora los siguientes KPIs:

| KPI | Campo / cálculo | Agregación |
|---|---|---|
| Valor ventas TY | `sales_value_actual` | `SUM` |
| Unidades vendidas TY | `units_value_actual` | `SUM` |
| Variación YoY valor % | `(SUM(sales_value_actual) - SUM(sales_value_anterior)) / SUM(sales_value_anterior)` | Cálculo agregado |
| Variación YoY unidades % | `(SUM(units_value_actual) - SUM(units_value_anterior)) / SUM(units_value_anterior)` | Cálculo agregado |
| % contribución TY | `pct_contribucion_2025` | `MAX` |
| Venta media semanal TY | `venta_media_semanal_2025` | `MAX` |
| Unidades medias semanales TY | `unidades_medias_semanales_2025` | `MAX` |
| Ranking 2025 | `ranking_2025` | `MAX` |

La variación YoY se calcula como una métrica agregada para evitar errores derivados de sumar porcentajes fila a fila.

#### 🧾 **Ficha técnica del producto**

La ficha técnica resume los principales atributos del producto seleccionado:

- imagen del producto;
- marca;
- proveedor;
- categoría;
- subcategoría o familia/formato de categoría;
- perfil de comportamiento;
- clasificación estratégica;
- ranking 2025.

Esta ficha permite contextualizar el producto antes de analizar sus métricas comerciales.

#### 📈 **Evolución ventas TY vs LY del producto**

Se incluye un gráfico de serie temporal para comparar la evolución semanal de las ventas del producto seleccionado frente al año anterior.

- Dimensión temporal: `week_start_date`
- Métrica TY: `SUM(sales_value_actual)`
- Métrica LY: `SUM(sales_value_anterior)`
- Filtros: `year_actual = 2025`, `escenario_actual = Real`

Se utiliza una fecha real como dimensión temporal para asegurar el orden cronológico correcto de las semanas.

#### 📊 **Comparativa del producto frente a su categoría**

Para contextualizar el rendimiento del producto, se incorpora una comparativa anual frente a su categoría.

La vista `vw_detalle_producto` incluye campos de benchmark calculados en BigQuery:

| Campo | Descripción |
|---|---|
| `sales_value_2025` | Ventas anuales del producto |
| `media_sales_value_categoria_2025` | Media de ventas de los productos de la misma categoría |
| `top_sales_value_categoria_2025` | Ventas del producto líder de la categoría |
| `top_producto_categoria_2025` | Nombre del producto líder de la categoría |
| `ratio_vs_media_categoria_2025` | Relación entre ventas del producto y media de la categoría |
| `ratio_vs_top_categoria_2025` | Relación entre ventas del producto y top de la categoría |

El gráfico compara:

- producto seleccionado;
- media de su categoría;
- top de su categoría.

Esta visualización permite identificar si el producto se sitúa por encima o por debajo de la media de su categoría y qué distancia mantiene respecto al producto líder.

#### 🏢 **Ficha del proveedor asociado**

La página de detalle incorpora también una ficha básica del proveedor principal del producto seleccionado.

Los campos incluidos son:

- proveedor principal;
- tipo de proveedor;
- lead time;
- pedido mínimo.

Además, se añade un botón de navegación hacia una página accesoria de análisis estratégico de proveedores.

Esta decisión permite mantener la página de producto centrada en el análisis individual del catálogo, pero ofrece una vía para profundizar en el proveedor cuando sea necesario.

#### 🔁 **Navegación hacia el análisis de proveedores**

Se incorpora un botón:

`Ver análisis del proveedor`

Este botón dirige a una página accesoria dedicada al análisis estratégico de proveedores.

La página de proveedores se plantea como una vista complementaria, no como parte de la navegación principal del dashboard. Su objetivo será analizar el peso comercial, prioridad y señales de riesgo de cada proveedor.

#### **5.2.1. Vista `vw_detalle_producto`**.

Para desarrollar la página de detalle de producto en Data Studio se crea la vista `vw_detalle_producto`.

Esta vista actúa como fuente principal de consumo para todos los elementos de la página:

- ficha técnica del producto;
- KPIs anuales del producto;
- evolución semanal TY vs LY;
- comparativa frente a la categoría;
- imagen del producto;
- información básica del proveedor asociado.

La decisión de crear una vista específica responde a una necesidad práctica: evitar el uso de múltiples fuentes de datos en Data Studio. En pruebas previas se comprobó que trabajar con varias vistas y combinaciones de datos podía generar problemas de filtrado, duplicidades o agregaciones incorrectas. Por ello, se opta por trasladar la lógica de preparación a BigQuery y dejar Data Studio principalmente como capa de visualización.

```sql

CREATE OR REPLACE VIEW `corepulse-sales-analytics.corepulse_analytics.vw_detalle_producto` AS

WITH producto_atributos AS (
    SELECT
        product_id,

        ANY_VALUE(nombre) AS nombre,
        ANY_VALUE(marca) AS marca,

        ANY_VALUE(category_id) AS category_id,
        ANY_VALUE(categoria) AS categoria,
        ANY_VALUE(familia_categoria) AS familia_categoria,
        ANY_VALUE(formato_categoria) AS formato_categoria,

        ANY_VALUE(provider_id) AS provider_id,
        ANY_VALUE(proveedor) AS proveedor,
        ANY_VALUE(pais) AS pais,
        ANY_VALUE(ccaa) AS ccaa,
        ANY_VALUE(tipo_proveedor) AS tipo_proveedor,
        ANY_VALUE(lead_time_dias_sim) AS lead_time_dias_sim,
        ANY_VALUE(pedido_minimo_sim) AS pedido_minimo_sim,

        ANY_VALUE(cluster_final) AS cluster_final,
        ANY_VALUE(perfil_comportamiento) AS perfil_comportamiento,
        ANY_VALUE(unit_sale_price_reference) AS unit_sale_price_reference

    FROM `corepulse-sales-analytics.corepulse_analytics.vw_ventas_base`
    GROUP BY product_id
),

ventas_2025 AS (
    SELECT
        product_id,

        SUM(sales_units) AS units_value_2025,
        SUM(sales_value_estimated) AS sales_value_2025,

        SUM(sales_units_adjusted) AS units_value_adjusted_2025,
        SUM(sales_value_estimated_adjusted) AS sales_value_adjusted_2025,

        COUNT(DISTINCT week_number_business) AS n_semanas_2025,

        SAFE_DIVIDE(
            SUM(sales_value_estimated),
            COUNT(DISTINCT week_number_business)
        ) AS venta_media_semanal_2025,

        SAFE_DIVIDE(
            SUM(sales_units),
            COUNT(DISTINCT week_number_business)
        ) AS unidades_medias_semanales_2025,

        SAFE_DIVIDE(
            SUM(sales_value_estimated),
            SUM(sales_units)
        ) AS precio_medio_venta_2025

    FROM `corepulse-sales-analytics.corepulse_analytics.vw_ventas_base`
    WHERE year_label = 2025
    GROUP BY product_id
),

ventas_2024 AS (
    SELECT
        product_id,

        SUM(sales_units) AS units_value_2024,
        SUM(sales_value_estimated) AS sales_value_2024,

        SUM(sales_units_adjusted) AS units_value_adjusted_2024,
        SUM(sales_value_estimated_adjusted) AS sales_value_adjusted_2024

    FROM `corepulse-sales-analytics.corepulse_analytics.vw_ventas_base`
    WHERE year_label = 2024
    GROUP BY product_id
),

forecast_2026 AS (
    SELECT
        product_id,

        SUM(forecast_base_units) AS forecast_base_units_2026,
        SUM(forecast_base_value_estimated) AS forecast_base_value_2026,

        SUM(forecast_optimista_units) AS forecast_optimista_units_2026,
        SUM(forecast_optimista_value_estimated) AS forecast_optimista_value_2026,

        SUM(forecast_pesimista_units) AS forecast_pesimista_units_2026,
        SUM(forecast_pesimista_value_estimated) AS forecast_pesimista_value_2026

    FROM `corepulse-sales-analytics.corepulse_analytics.vw_ventas_base`
    WHERE year_label = 2026
    GROUP BY product_id
),

ranking_2025 AS (
    SELECT
        product_id,
        ranking_producto AS ranking_2025,
        pct_contribucion AS pct_contribucion_2025,
        pct_contribucion_acumulada AS pct_contribucion_acumulada_2025
    FROM `corepulse-sales-analytics.corepulse_analytics.vw_ranking_productos_anual_escenario`
    WHERE year_label = 2025
      AND escenario = 'Real'
),

ranking_forecast_2026 AS (
    SELECT
        product_id,
        ranking_producto AS ranking_forecast_base_2026
    FROM `corepulse-sales-analytics.corepulse_analytics.vw_ranking_productos_anual_escenario`
    WHERE year_label = 2026
      AND escenario = 'Forecast base'
),

campania_top_2025 AS (
    SELECT
        product_id,
        COALESCE(campaign_name_primary, 'Sin campaña') AS campania_top_2025,
        SUM(sales_value_estimated) AS sales_value_campania_top_2025,

        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY SUM(sales_value_estimated) DESC
        ) AS rn

    FROM `corepulse-sales-analytics.corepulse_analytics.vw_ventas_base`
    WHERE year_label = 2025
    GROUP BY
        product_id,
        COALESCE(campaign_name_primary, 'Sin campaña')
),

producto_resumen AS (
    SELECT
        p.product_id,
        p.nombre,
        p.marca,

        p.category_id,
        p.categoria,
        p.familia_categoria,
        p.formato_categoria,

        p.provider_id,
        p.proveedor,
        p.pais,
        p.ccaa,
        p.tipo_proveedor,
        p.lead_time_dias_sim,
        p.pedido_minimo_sim,

        p.cluster_final,
        p.perfil_comportamiento,
        p.unit_sale_price_reference,

        cls.clasificacion_producto,
        cls.orden_clasificacion_producto,
        cls.descripcion_clasificacion,

        v25.units_value_2025,
        v25.sales_value_2025,
        v25.units_value_adjusted_2025,
        v25.sales_value_adjusted_2025,
        v25.n_semanas_2025,
        v25.venta_media_semanal_2025,
        v25.unidades_medias_semanales_2025,
        v25.precio_medio_venta_2025,

        v24.units_value_2024,
        v24.sales_value_2024,
        v24.units_value_adjusted_2024,
        v24.sales_value_adjusted_2024,

        SAFE_DIVIDE(
            v25.sales_value_2025 - v24.sales_value_2024,
            v24.sales_value_2024
        ) AS var_value_yoy_pct,

        SAFE_DIVIDE(
            v25.units_value_2025 - v24.units_value_2024,
            v24.units_value_2024
        ) AS var_units_yoy_pct,

        r25.ranking_2025,
        r25.pct_contribucion_2025,
        r25.pct_contribucion_acumulada_2025,

        f26.forecast_base_units_2026,
        f26.forecast_base_value_2026,
        f26.forecast_optimista_units_2026,
        f26.forecast_optimista_value_2026,
        f26.forecast_pesimista_units_2026,
        f26.forecast_pesimista_value_2026,

        rf26.ranking_forecast_base_2026,

        SAFE_DIVIDE(
            f26.forecast_base_value_2026 - v25.sales_value_2025,
            v25.sales_value_2025
        ) AS var_forecast_base_2026_pct,

        ct.campania_top_2025,
        ct.sales_value_campania_top_2025

    FROM producto_atributos p

    LEFT JOIN ventas_2025 v25
        ON p.product_id = v25.product_id

    LEFT JOIN ventas_2024 v24
        ON p.product_id = v24.product_id

    LEFT JOIN forecast_2026 f26
        ON p.product_id = f26.product_id

    LEFT JOIN ranking_2025 r25
        ON p.product_id = r25.product_id

    LEFT JOIN ranking_forecast_2026 rf26
        ON p.product_id = rf26.product_id

    LEFT JOIN campania_top_2025 ct
        ON p.product_id = ct.product_id
       AND ct.rn = 1

    LEFT JOIN `corepulse-sales-analytics.corepulse_analytics.vw_clasificacion_producto_estrategica` cls
        ON p.product_id = cls.product_id
),

benchmark_categoria_2025 AS (
    SELECT
        category_id,
        categoria,

        COUNT(DISTINCT product_id) AS n_productos_categoria,

        AVG(sales_value_2025) AS media_sales_value_categoria_2025,
        AVG(units_value_2025) AS media_units_value_categoria_2025,

        MAX(sales_value_2025) AS top_sales_value_categoria_2025,
        MAX(units_value_2025) AS top_units_value_categoria_2025

    FROM producto_resumen
    GROUP BY
        category_id,
        categoria
),

top_producto_categoria_2025 AS (
    SELECT
        category_id,
        product_id AS top_product_id_categoria_2025,
        nombre AS top_producto_categoria_2025,
        sales_value_2025 AS top_sales_value_producto_categoria_2025,
        units_value_2025 AS top_units_value_producto_categoria_2025

    FROM producto_resumen

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY category_id
        ORDER BY sales_value_2025 DESC
    ) = 1
)

SELECT
    -- ==============================
    -- 1. IDENTIFICACIÓN PRODUCTO
    -- ==============================
    y.product_id,
    y.nombre,
    y.marca,
    y.categoria,
    y.proveedor,

    r.category_id,
    r.familia_categoria,
    r.formato_categoria,

    r.provider_id,
    r.pais,
    r.ccaa,
    r.tipo_proveedor,
    r.lead_time_dias_sim,
    r.pedido_minimo_sim,

    r.cluster_final,
    r.perfil_comportamiento,
    r.unit_sale_price_reference,

    -- ==============================
    -- 2. IMAGEN DEL PRODUCTO
    -- ==============================
    img.image_url,
    img.image_alt,

    -- ==============================
    -- 3. CLASIFICACIÓN ESTRATÉGICA
    -- ==============================
    r.clasificacion_producto,
    r.orden_clasificacion_producto,
    r.descripcion_clasificacion,

    -- ==============================
    -- 4. CAMPOS TEMPORALES / YoY
    -- ==============================
    y.year_anterior,
    y.year_actual,

    y.escenario_anterior,
    y.escenario_actual,

    y.week_number_business,

    y.year_week_key_anterior,
    y.year_week_key_actual,

    y.week_label_anterior,
    y.week_label_actual,

    y.week_start_date,
    y.week_end_date,

    y.is_partial_week,
    y.is_campaign_week,
    y.campaign_name_primary,
    y.campaign_group_primary,

    y.periodo_comparativo,

    -- ==============================
    -- 5. MÉTRICAS SEMANALES TY / LY
    -- Estas sí se agregan con SUM en Looker Studio
    -- ==============================
    y.units_value_anterior,
    y.units_value_actual,

    y.sales_value_anterior,
    y.sales_value_actual,

    y.units_value_adjusted_anterior,
    y.units_value_adjusted_actual,

    y.sales_value_adjusted_anterior,
    y.sales_value_adjusted_actual,

    y.var_units_abs,
    y.var_units_pct,

    y.var_value_abs,
    y.var_value_pct,

    y.var_units_adjusted_abs,
    y.var_units_adjusted_pct,

    y.var_value_adjusted_abs,
    y.var_value_adjusted_pct,

    -- ==============================
    -- 6. MÉTRICAS ANUALES 2025
    -- OJO: están repetidas por semana.
    -- En Looker Studio usar MAX/MIN, no SUM.
    -- ==============================
    r.units_value_2025,
    r.sales_value_2025,

    r.units_value_adjusted_2025,
    r.sales_value_adjusted_2025,

    r.n_semanas_2025,
    r.venta_media_semanal_2025,
    r.unidades_medias_semanales_2025,
    r.precio_medio_venta_2025,

    r.units_value_2024,
    r.sales_value_2024,

    r.var_value_yoy_pct,
    r.var_units_yoy_pct,

    -- ==============================
    -- 7. RANKING Y CONTRIBUCIÓN
    -- También usar MAX/MIN en Looker Studio
    -- ==============================
    r.ranking_2025,
    r.pct_contribucion_2025,
    r.pct_contribucion_acumulada_2025,

    -- ==============================
    -- 8. FORECAST 2026
    -- También campos anuales repetidos
    -- ==============================
    r.forecast_base_units_2026,
    r.forecast_base_value_2026,

    r.forecast_optimista_units_2026,
    r.forecast_optimista_value_2026,

    r.forecast_pesimista_units_2026,
    r.forecast_pesimista_value_2026,

    r.ranking_forecast_base_2026,
    r.var_forecast_base_2026_pct,

    -- ==============================
    -- 9. CAMPAÑA TOP DEL PRODUCTO
    -- ==============================
    r.campania_top_2025,
    r.sales_value_campania_top_2025,

    -- ==============================
    -- 10. BENCHMARK FRENTE A CATEGORÍA
    -- Campos anuales repetidos.
    -- En Looker Studio usar MAX/MIN.
    -- ==============================
    bcat.n_productos_categoria,

    bcat.media_sales_value_categoria_2025,
    bcat.media_units_value_categoria_2025,

    bcat.top_sales_value_categoria_2025,
    bcat.top_units_value_categoria_2025,

    tcat.top_product_id_categoria_2025,
    tcat.top_producto_categoria_2025,
    tcat.top_sales_value_producto_categoria_2025,
    tcat.top_units_value_producto_categoria_2025,

    SAFE_DIVIDE(
        r.sales_value_2025,
        bcat.media_sales_value_categoria_2025
    ) AS ratio_vs_media_categoria_2025,

    SAFE_DIVIDE(
        r.sales_value_2025,
        bcat.top_sales_value_categoria_2025
    ) AS ratio_vs_top_categoria_2025,

    SAFE_DIVIDE(
        r.units_value_2025,
        bcat.media_units_value_categoria_2025
    ) AS ratio_units_vs_media_categoria_2025,

    SAFE_DIVIDE(
        r.units_value_2025,
        bcat.top_units_value_categoria_2025
    ) AS ratio_units_vs_top_categoria_2025

FROM `corepulse-sales-analytics.corepulse_analytics.vw_ventas_yoy_producto_semana` y

LEFT JOIN producto_resumen r
    ON y.product_id = r.product_id

LEFT JOIN `corepulse-sales-analytics.corepulse_dw.dim_producto_imagen` img
    ON y.product_id = img.product_id

LEFT JOIN benchmark_categoria_2025 bcat
    ON r.category_id = bcat.category_id

LEFT JOIN top_producto_categoria_2025 tcat
    ON r.category_id = tcat.category_id;

🎯 **Objetivo de la vista**

El objetivo de `vw_detalle_producto` es centralizar en una única fuente toda la información necesaria para analizar un producto individual.

La vista debe permitir responder a preguntas como:

- ¿cuánto ha vendido el producto en 2025?
- ¿cómo evoluciona frente a 2024?
- ¿qué posición ocupa en el ranking del catálogo?
- ¿qué peso tiene sobre las ventas totales?
- ¿cuál es su clasificación estratégica?
- ¿cuál es su perfil de comportamiento?
- ¿cómo se compara con la media y el producto líder de su categoría?
- ¿qué proveedor está asociado al producto?
- ¿cuál es su imagen asociada?

De esta forma, la página de detalle puede funcionar a partir de un selector principal de producto y mantener una respuesta coherente en todos sus elementos visuales.

🔎 **Grano de la vista**

La vista mantiene un grano semanal:

`1 fila = 1 producto + 1 semana + 1 año actual + 1 escenario actual`

Esto significa que cada producto aparece repetido por semana dentro del año analizado.

Este grano se conserva porque la página de detalle necesita mostrar una evolución temporal del producto seleccionado, comparando las ventas TY frente a LY.

Sin embargo, la vista también incorpora métricas anuales del producto. Estas métricas se repiten en todas las filas semanales del mismo producto, por lo que deben tratarse con cuidado en Data Studio.

🧩 **Fuentes utilizadas**

La vista `vw_detalle_producto` se construye a partir de varias fuentes ya preparadas previamente en BigQuery:

| Fuente | Uso dentro de `vw_detalle_producto` |
|---|---|
| `vw_ventas_yoy_producto_semana` | Base semanal TY vs LY del producto |
| `vw_ventas_base` | Cálculo de atributos, ventas anuales, forecast y métricas auxiliares |
| `vw_ranking_productos_anual_escenario` | Ranking anual, contribución y ranking forecast |
| `vw_clasificacion_producto_estrategica` | Clasificación estratégica del producto |
| `dim_producto_imagen` | Imagen dinámica del producto |

La vista final combina estas fuentes mediante CTEs internos y `LEFT JOIN`, manteniendo como base principal la vista YoY semanal.

**CTEs principales de la vista**

La vista se organiza mediante distintos bloques lógicos.

1. **`producto_atributos`**

Este bloque obtiene los atributos descriptivos del producto desde `vw_ventas_base`.

Incluye campos como:

- `product_id`
- `nombre`
- `marca`
- `category_id`
- `categoria`
- `familia_categoria`
- `formato_categoria`
- `provider_id`
- `proveedor`
- `tipo_proveedor`
- `lead_time_dias_sim`
- `pedido_minimo_sim`
- `cluster_final`
- `perfil_comportamiento`
- `unit_sale_price_reference`

Se utiliza `ANY_VALUE()` porque estos atributos son constantes para cada producto dentro del modelo.

2. **`ventas_2025`**

Este bloque calcula las métricas anuales reales del producto para 2025.

Incluye:

- unidades vendidas en 2025;
- ventas estimadas en valor;
- ventas ajustadas;
- número de semanas con datos;
- venta media semanal;
- unidades medias semanales;
- precio medio de venta.

Estas métricas se calculan a nivel producto y posteriormente se incorporan a la vista semanal.

3. **`ventas_2024`**

Este bloque calcula las métricas equivalentes para 2024, necesarias para obtener la variación YoY del producto.

Incluye:

- unidades vendidas en 2024;
- ventas estimadas en valor en 2024;
- ventas ajustadas en 2024.

Estos datos permiten comparar el rendimiento del producto entre 2025 y 2024.

4. **`forecast_2026`**

Este bloque agrega la previsión 2026 del producto para los tres escenarios:

- forecast base;
- forecast optimista;
- forecast pesimista.

Aunque la página de detalle se centra en 2025, se incorpora el forecast 2026 para enriquecer el análisis del producto con una señal futura.

**5. `ranking_2025`**

Este bloque incorpora el ranking anual del producto en 2025 y su contribución sobre las ventas totales.

Campos principales:

- `ranking_2025`
- `pct_contribucion_2025`
- `pct_contribucion_acumulada_2025`

Estos campos permiten situar el producto dentro del catálogo global.

6. **`ranking_forecast_2026`**

Este bloque incorpora el ranking previsto del producto en el escenario base de 2026.

El objetivo es comparar la posición actual del producto con su posición esperada en el forecast base.

7. **`campania_top_2025`**

Este bloque identifica la campaña con mayor volumen de ventas para cada producto en 2025.

Para ello se agrupan las ventas por producto y campaña, y se utiliza `ROW_NUMBER()` para seleccionar la campaña con mayor facturación.

El campo resultante permite saber en qué contexto comercial destaca más cada producto.

8. **`producto_resumen`**

Este bloque consolida todos los cálculos anuales del producto en una única estructura.

Aquí se unen:

- atributos del producto;
- ventas 2025;
- ventas 2024;
- forecast 2026;
- ranking 2025;
- ranking forecast 2026;
- campaña principal;
- clasificación estratégica.

También se calculan métricas derivadas como:

- variación YoY en valor;
- variación YoY en unidades;
- variación forecast base 2026 frente a ventas 2025.


9. **`benchmark_categoria_2025`**

Este bloque calcula métricas de referencia por categoría para 2025.

El objetivo es poder comparar el producto seleccionado con su contexto competitivo más cercano: su propia categoría.

Se calculan:

- número de productos en la categoría;
- media de ventas de la categoría;
- media de unidades vendidas de la categoría;
- ventas máximas dentro de la categoría;
- unidades máximas dentro de la categoría.

Estas métricas permiten construir el gráfico de comparación:

- producto seleccionado;
- media de su categoría;
- top de su categoría.

**10. `top_producto_categoria_2025`**

Este bloque identifica el producto líder en ventas dentro de cada categoría.

Se utiliza `ROW_NUMBER()` particionado por `category_id` y ordenado por ventas 2025 descendentes.

El resultado permite incorporar a la vista:

- identificador del producto top de la categoría;
- nombre del producto top;
- ventas del producto top;
- unidades del producto top.

🖼️ **Integración de imágenes**

La imagen del producto se incorpora mediante la tabla `dim_producto_imagen`.

Esta tabla contiene:

| Campo | Descripción |
|---|---|
| `product_id` | Identificador del producto |
| `image_url` | URL pública de la imagen |
| `image_alt` | Texto alternativo de la imagen |

La unión se realiza mediante:

`product_id`

En Data Studio se crea posteriormente un campo calculado con la función:

```sql
IMAGE(image_url, image_alt)

Esto permite mostrar una imagen dinámica en la ficha del producto, que cambia al seleccionar un producto distinto.


📊 **Campos principales incorporados en la vista**

La vista final incorpora distintos grupos de campos.

###### Identificación del producto

- `product_id`
- `nombre`
- `marca`
- `categoria`
- `proveedor`
- `category_id`
- `provider_id`

###### Atributos del producto

- `familia_categoria`
- `formato_categoria`
- `cluster_final`
- `perfil_comportamiento`
- `unit_sale_price_reference`

###### Imagen

- `image_url`
- `image_alt`

###### Clasificación estratégica

- `clasificacion_producto`
- `orden_clasificacion_producto`
- `descripcion_clasificacion`

###### Campos temporales

- `year_actual`
- `year_anterior`
- `week_number_business`
- `week_start_date`
- `week_end_date`
- `week_label_actual`
- `periodo_comparativo`

###### Métricas semanales

- `sales_value_actual`
- `sales_value_anterior`
- `units_value_actual`
- `units_value_anterior`
- `sales_value_adjusted_actual`
- `sales_value_adjusted_anterior`

###### Métricas anuales

- `sales_value_2025`
- `units_value_2025`
- `venta_media_semanal_2025`
- `unidades_medias_semanales_2025`
- `precio_medio_venta_2025`
- `ranking_2025`
- `pct_contribucion_2025`

###### Forecast 2026

- `forecast_base_value_2026`
- `forecast_optimista_value_2026`
- `forecast_pesimista_value_2026`
- `ranking_forecast_base_2026`
- `var_forecast_base_2026_pct`

###### Benchmark de categoría

- `media_sales_value_categoria_2025`
- `top_sales_value_categoria_2025`
- `top_producto_categoria_2025`
- `ratio_vs_media_categoria_2025`
- `ratio_vs_top_categoria_2025`

⚠️ **Tratamiento de agregaciones en Data Studio**

Debido a que la vista mantiene grano semanal, algunas métricas están repetidas por cada semana del producto.

Por ello, en Data Studio se debe distinguir entre métricas semanales y métricas anuales repetidas.

| Tipo de métrica | Ejemplos | Agregación correcta |
|---|---|---|
| Métricas semanales | `sales_value_actual`, `units_value_actual` | `SUM` |
| Métricas anuales repetidas | `ranking_2025`, `pct_contribucion_2025`, `venta_media_semanal_2025` | `MAX` o `MIN` |
| Ratios anuales | `var_value_yoy_pct`, `var_forecast_base_2026_pct` | `MAX` |
| Dimensiones | `clasificacion_producto`, `perfil_comportamiento`, `proveedor` | Dimensión |

Esta decisión evita errores como sumar porcentajes o rankings repetidos por semana.

📌 **Justificación metodológica**

La creación de `vw_detalle_producto` permite resolver una de las principales limitaciones detectadas durante el desarrollo del dashboard: la dificultad de mantener filtros coherentes cuando se utilizan varias fuentes de datos en Data Studio.

En lugar de conectar la ficha de producto a una vista, los KPIs a otra y las visualizaciones temporales a otra distinta, se centraliza toda la información en una única vista de consumo.

Esta decisión aporta varias ventajas:

- mejora la coherencia de filtros;
- reduce la necesidad de combinaciones de datos;
- evita duplicidades por uniones incorrectas en Data Studio;
- facilita el mantenimiento del dashboard;
- permite documentar claramente qué campos son semanales y cuáles son anuales repetidos;
- mantiene la lógica compleja en BigQuery, dejando Data Studio como capa de visualización.

La vista `vw_detalle_producto` se convierte así en la base analítica de la página de detalle de producto.

> En resumen, `vw_detalle_producto` permite construir una página de detalle completa y coherente, combinando análisis temporal, indicadores anuales, atributos descriptivos, imagen del producto y comparación frente a la categoría, sin recurrir a combinaciones de datos dentro de Data Studio.

#### **5.2.2. Decisiones metodológicas**.

Durante el desarrollo de la página de detalle se toman varias decisiones relevantes:

1. **Usar una única fuente principal para la página**

   Se decide utilizar `vw_detalle_producto` como fuente común para evitar problemas de filtros cruzados entre varias vistas en Looker Studio.

2. **Mantener grano semanal en la vista**

   El grano semanal permite construir gráficos de evolución TY vs LY del producto seleccionado.

3. **Incorporar métricas anuales repetidas**

   Campos como ranking, contribución, forecast o venta media semanal se incorporan a la vista aunque estén repetidos por semana. En Looker Studio se muestran mediante agregaciones como `MAX` o `MIN`.

4. **Evitar combinaciones de datos innecesarias**

   En lugar de combinar varias fuentes dentro de Looker Studio, se traslada la lógica de enriquecimiento a BigQuery.

5. **Separar el análisis de producto y proveedor**

   La página de detalle se centra en el producto seleccionado. El análisis profundo del proveedor se deriva a una página accesoria específica.

### **5.3. Desarrollo de la página de análisis estratégico de proveedores en Data Studio.**

#### 🎯 **Objetivo de la página**

La página de análisis estratégico de proveedores se diseña como una vista complementaria a la página de detalle de producto.

Mientras que la página anterior permite analizar el comportamiento de un producto concreto, esta página se centra en evaluar el papel del proveedor dentro del negocio, considerando tanto su peso comercial como las señales de riesgo asociadas a los productos que suministra.

El objetivo principal de esta página es responder a preguntas como:

- qué peso comercial tiene cada proveedor;
- qué posición ocupa dentro del ranking de proveedores;
- qué porcentaje de las ventas totales representa;
- cuántos productos asociados presentan señales de riesgo;
- qué porcentaje de sus ventas está vinculado a productos en riesgo;
- cuál es su nivel de dependencia comercial;
- cuál es su prioridad de revisión;
- qué productos explican su comportamiento;
- cómo evolucionan sus ventas frente al año anterior.

Esta página se plantea como una vista de apoyo para la toma de decisiones, especialmente útil para identificar proveedores relevantes que requieren seguimiento comercial u operativo.

#### 🧩 **Fuente principal de datos**

Para mantener la coherencia de filtros y evitar combinaciones de datos innecesarias en Data Studio, se crea una única vista principal para esta página:

`vw_detalle_proveedor`

Esta vista se utiliza como fuente común para:

- ficha del proveedor;
- KPIs estratégicos;
- evolución semanal TY vs LY;
- distribución de ventas por clasificación estratégica;
- tabla de productos asociados al proveedor.

La vista mantiene un grano detallado:

`1 fila = 1 proveedor + 1 producto + 1 semana`

Este grano permite combinar dos necesidades:

1. analizar la evolución semanal de las ventas del proveedor;
2. conservar información agregada del proveedor, repetida en cada fila, para construir KPIs y fichas descriptivas.

Por tanto, igual que ocurría en la vista de detalle de producto, es necesario diferenciar entre métricas semanales y métricas anuales repetidas.

#### 📌 **Dependencia comercial del proveedor**

La dependencia comercial mide cuánto depende el negocio de un proveedor desde el punto de vista de ventas.

No debe confundirse con la prioridad del proveedor.

- **Dependencia comercial**: indica importancia o peso comercial.
- **Prioridad del proveedor**: indica necesidad de revisión o seguimiento.

Un proveedor puede tener alta dependencia comercial y no presentar señales críticas de riesgo. Del mismo modo, un proveedor con dependencia baja puede tener productos problemáticos, pero su impacto global sobre el negocio será menor.

La dependencia comercial se calcula a partir del ranking del proveedor en 2025.

| Condición | Dependencia comercial |
|---|---|
| Proveedor dentro del top 25% por ventas | Alta |
| Proveedor entre el top 25% y top 50% | Media |
| Resto de proveedores | Baja |

Esta regla permite clasificar a los proveedores según su peso relativo dentro del conjunto, sin fijar umbrales absolutos de ventas.

#### 🚦 **Prioridad del proveedor**

La prioridad del proveedor indica si un proveedor requiere seguimiento o revisión desde una perspectiva comercial y operativa.

A diferencia de la dependencia comercial, la prioridad no solo tiene en cuenta el volumen de ventas, sino también las señales de riesgo asociadas a los productos del proveedor.

Para calcularla se consideran los siguientes factores:

- dependencia comercial;
- porcentaje de ventas asociadas a productos en riesgo;
- porcentaje de ventas asociadas a productos líderes en riesgo;
- existencia de productos rezagados;
- lead time elevado respecto al resto de proveedores.

La lógica final diferencia tres niveles:

| Prioridad | Interpretación |
|---|---|
| Alta | Proveedor relevante con señales claras de riesgo o exposición elevada |
| Media | Proveedor con señales moderadas de seguimiento |
| Baja | Proveedor sin señales relevantes de prioridad |

La prioridad alta se reserva para casos donde existe una combinación de impacto comercial y riesgo relevante. Esto evita clasificar como prioritarios a todos los proveedores que simplemente tengan algún producto rezagado.


##### **Reglas de prioridad**

La prioridad del proveedor se determina de forma aproximada mediante las siguientes reglas:

| Condición | Prioridad |
|---|---|
| Alta dependencia comercial y ventas relevantes asociadas a productos líderes en riesgo | Alta |
| Alta dependencia comercial y concentración muy elevada de ventas en productos de riesgo o rezagados | Alta |
| Alta dependencia comercial, lead time elevado y elevada exposición a productos de riesgo | Alta |
| Ventas asociadas a productos líderes en riesgo, pero sin cumplir criterios de prioridad alta | Media |
| Alta dependencia comercial con exposición moderada a productos en riesgo o rezagados | Media |
| Dependencia media con elevada concentración de ventas en productos en riesgo o rezagados | Media |
| Dependencia media con lead time elevado | Media |
| Resto de proveedores | Baja |

Esta lógica se ajustó durante el desarrollo, ya que una primera versión clasificaba a todos los proveedores como prioridad alta al considerar de forma demasiado amplia los productos rezagados. La regla final separa mejor los productos `Líder en riesgo` de los productos `Rezagado`, considerando el impacto comercial real de cada grupo.

#### 🧮 **KPIs principales de la página**

La página incorpora los siguientes KPIs para el proveedor seleccionado:

| KPI | Campo | Agregación en Data Studio | Interpretación |
|---|---|---|---|
| Ventas TY proveedor | `sales_value_proveedor_2025` | `MAX` | Ventas totales del proveedor en 2025 |
| % contribución proveedor | `pct_contribucion_proveedor_2025` | `MAX` | Peso del proveedor sobre las ventas totales |
| Productos en riesgo | `n_productos_en_riesgo` | `MAX` | Nº de productos del proveedor clasificados como riesgo o rezagados |
| % ventas en riesgo | `pct_ventas_en_riesgo` | `MAX` | Peso de las ventas del proveedor asociadas a productos en riesgo |
| Ranking proveedor 2025 | `ranking_proveedor_2025` | `MIN` | Posición del proveedor en el ranking de ventas 2025 |

Como la vista tiene grano proveedor-producto-semana, estos campos están repetidos en múltiples filas. Por ello, se muestran con agregaciones como `MAX` o `MIN`, evitando sumar valores que ya están calculados a nivel proveedor.

#### 🧾 **Ficha del proveedor**

La página incluye una ficha descriptiva del proveedor seleccionado.

Campos incluidos:

- nombre del proveedor;
- tipo de proveedor;
- ubicación;
- lead time;
- pedido mínimo;
- dependencia comercial;
- prioridad del proveedor;
- motivo de prioridad.

Esta ficha permite interpretar los KPIs dentro de un contexto operativo y comercial.

Además, se aplican colores diferenciados para facilitar la lectura:

- dependencia comercial alta/media/baja;
- prioridad alta/media/baja.

La prioridad se muestra como alerta visual, mientras que la dependencia comercial representa el peso del proveedor dentro del negocio.

#### 📈 **Evolución ventas TY vs LY del proveedor**

Se incluye un gráfico de evolución semanal para comparar las ventas del proveedor en 2025 frente a las ventas equivalentes del año anterior.

- Dimensión temporal: `week_start_date`
- Métrica TY: `SUM(sales_value_actual)`
- Métrica LY: `SUM(sales_value_anterior)`

En este gráfico se utiliza una combinación visual de línea y barras:

- línea para representar las ventas TY 2025;
- barras para representar las ventas LY 2024.

Esta decisión mejora la lectura visual, ya que permite distinguir claramente el comportamiento actual frente al periodo de comparación.

#### 📊 **Ventas por clasificación estratégica**

La página incorpora un gráfico de barras horizontales que muestra la distribución de las ventas del proveedor según la clasificación estratégica de sus productos.

- Dimensión: `clasificacion_producto`
- Métrica: `SUM(sales_value_actual)`

El objetivo es identificar si las ventas del proveedor proceden principalmente de:

- productos líderes consolidados;
- productos líderes en riesgo;
- productos emergentes;
- productos rezagados.

Esta visualización ayuda a justificar el nivel de prioridad asignado al proveedor. Por ejemplo, un proveedor con alta dependencia comercial y un peso elevado en productos líderes en riesgo o rezagados puede requerir seguimiento prioritario.

#### 📋 **Tabla de productos asociados al proveedor**

Se incluye una tabla con los principales productos asociados al proveedor seleccionado.

Campos utilizados:

- nombre del producto;
- categoría;
- clasificación estratégica;
- perfil de comportamiento;
- ventas TY;
- unidades TY;
- variación YoY;
- forecast base 2026;
- ranking del producto.

La tabla se ordena por ventas TY de forma descendente, para mostrar primero los productos con mayor impacto comercial dentro del proveedor.

Esta tabla permite entender qué productos explican el peso, riesgo o prioridad del proveedor seleccionado.

#### 🔁 **Navegación entre páginas**

La página de análisis estratégico de proveedores se plantea como una página accesoria.

Se accede a ella desde la página de detalle de producto mediante un botón:

`Ver análisis del proveedor`

Además, la página de proveedores incorpora un botón de retorno:

`Volver al detalle de producto`

Esto permite mantener una navegación fluida entre el análisis del producto y el análisis de su proveedor asociado.

#### **5.3.1. Vista `vw_detalle_proveedor`**.

La vista `vw_detalle_proveedor` se construye a partir de la vista `vw_detalle_producto`, que ya contiene información semanal del producto, clasificación estratégica, ranking, contribución y métricas anuales.

A partir de esa información, se agregan los datos a nivel proveedor para calcular indicadores estratégicos.

La vista incorpora información de tres niveles:

| Nivel | Descripción |
|---|---|
| Proveedor | Datos identificativos, dependencia, prioridad, ranking y métricas agregadas |
| Producto | Productos asociados al proveedor y sus señales comerciales |
| Semana | Evolución semanal TY vs LY del proveedor |

Esta estructura permite que todos los elementos de la página respondan al selector de proveedor, manteniendo una fuente única en Data Studio.

**CTEs principales de la vista**

La vista se organiza mediante varios bloques lógicos.

1.**`producto_2025`**

Este bloque parte de `vw_detalle_producto` y obtiene una fila agregada por producto para el año 2025.

Incluye información como:

- identificador del producto;
- nombre del producto;
- marca;
- categoría;
- proveedor;
- clasificación estratégica;
- perfil de comportamiento;
- ventas 2025;
- unidades 2025;
- ranking del producto;
- contribución del producto;
- variación YoY;
- forecast base 2026.

Este bloque sirve como base para calcular los indicadores agregados por proveedor.

2. **`ranking_proveedor_2025`**

Este bloque incorpora el ranking anual del proveedor en 2025, procedente de la vista de ranking de proveedores.

Campos principales:

- `ranking_proveedor_2025`;
- `pct_contribucion_proveedor_2025`;
- `pct_contribucion_acumulada_proveedor_2025`;
- número total de proveedores;
- umbral top 25%;
- umbral top 50%.

Estos umbrales se utilizan posteriormente para determinar el nivel de dependencia comercial del proveedor.

3. **`proveedor_agregado`**

Este bloque calcula las métricas agregadas a nivel proveedor.

Entre ellas:

- número de productos asociados;
- ventas TY del proveedor;
- unidades TY del proveedor;
- número de productos por clasificación estratégica;
- número de productos en riesgo;
- ventas asociadas a productos en riesgo;
- ventas asociadas a productos líderes en riesgo;
- ventas asociadas a productos rezagados.

Se considera que un producto está en riesgo cuando pertenece a una de estas clasificaciones:

- `Líder en riesgo`;
- `Rezagado`.

No obstante, posteriormente se diferencia entre ambos tipos de riesgo para evitar que la prioridad del proveedor sea demasiado sensible a productos rezagados de bajo impacto.

4. **`top_producto_proveedor_2025`**

Este bloque identifica el producto con mayor volumen de ventas dentro de cada proveedor.

Para ello, se ordenan los productos de cada proveedor por ventas 2025 de forma descendente y se selecciona el primero mediante `ROW_NUMBER()`.

Este cálculo permite obtener:

- producto principal del proveedor;
- categoría del producto principal;
- ventas del producto principal;
- unidades del producto principal;
- concentración del proveedor en su producto principal.

La concentración del top producto se calcula como:

`ventas del producto principal / ventas totales del proveedor`

Este indicador permite detectar si el proveedor depende excesivamente de un único producto.

5. **`percentiles_proveedor`**

Este bloque calcula el percentil 75 del lead time simulado.

El objetivo es disponer de una referencia para identificar proveedores con plazos de entrega elevados en comparación con el resto.

Este valor se utiliza dentro de la lógica de prioridad del proveedor.

6. **`proveedor_base`**

Este bloque consolida las métricas calculadas previamente:

- métricas agregadas del proveedor;
- ranking del proveedor;
- contribución sobre ventas totales;
- producto principal;
- concentración del producto principal;
- porcentaje de ventas en riesgo;
- porcentaje de ventas asociadas a líderes en riesgo;
- porcentaje de ventas asociadas a productos rezagados;
- percentil 75 de lead time.

Aquí se calculan varios ratios clave:

| Campo | Descripción |
|---|---|
| `pct_ventas_en_riesgo` | Ventas del proveedor asociadas a productos en riesgo / ventas totales del proveedor |
| `pct_ventas_lider_riesgo` | Ventas asociadas a productos líderes en riesgo / ventas totales del proveedor |
| `pct_ventas_rezagado` | Ventas asociadas a productos rezagados / ventas totales del proveedor |
| `pct_concentracion_top_producto` | Ventas del producto principal / ventas totales del proveedor |

#### **5.3.2. Decisiones metodológicas**.



Durante el desarrollo de la página de proveedores se toman varias decisiones relevantes:

1. **Crear una vista específica para proveedores**

   Se crea `vw_detalle_proveedor` para centralizar toda la lógica necesaria en BigQuery y evitar combinaciones de datos complejas en Data Studio.

2. **Mantener un grano proveedor-producto-semana**

   Este grano permite analizar tanto la evolución temporal del proveedor como los productos que lo componen.

3. **Separar dependencia comercial y prioridad**

   La dependencia comercial mide el peso del proveedor en el negocio, mientras que la prioridad mide la necesidad de seguimiento o revisión.

4. **Refinar la lógica de prioridad**

   La primera versión clasificaba a todos los proveedores como prioridad alta. Se revisó la lógica para diferenciar mejor entre productos líderes en riesgo y productos rezagados, obteniendo una distribución más razonable entre prioridad alta, media y baja.

5. **No incluir un bloque global de alertas en esta página**

   Se valoró incluir un bloque adicional con el número total de proveedores a revisar y una tabla global de alertas. Finalmente se descartó para no sobrecargar la página y mantener el foco en el proveedor seleccionado.

6. **Usar una única fuente principal para la página**

   Todos los elementos de la página se construyen a partir de `vw_detalle_proveedor`, garantizando coherencia en los filtros y evitando inconsistencias.

#### **5.3.3. Resultado final**.



La página de análisis estratégico de proveedores permite evaluar de forma individual cada proveedor desde una perspectiva comercial y operativa.

La combinación de ficha, KPIs, evolución temporal, distribución por clasificación estratégica y tabla de productos asociados permite entender:

- cuánto pesa el proveedor en el negocio;
- qué riesgo presenta;
- qué productos explican su comportamiento;
- qué prioridad de seguimiento requiere.

Esta página complementa la página de detalle de producto y refuerza el carácter analítico del dashboard.

### **5.3. Desarrollo de la página de detalle de producto en Data Studio.**